# KV Retrieve Unified Analysis

This notebook replaces the split circuit / feature / neuron / algorithm notebooks with one canonical artifact.

It keeps all of the tracking machinery that was already built, but organizes it once in a single flow:

1. baseline prompt inspection
2. clean/corrupt behavior
3. activation patching, path patching, causal ablations
4. QK / OV analysis and circuit tracing
5. sparse autoencoder feature analysis
6. neuron-level analysis
7. register-style summary of the current internal program

The point is not to pretend this is already a solved algorithm notebook. The point is to have one place where every analysis object is visible, non-duplicated, and explained.


## Setup

This cell loads the model, dataset, saved SAE artifacts, and one anchor prompt used throughout the notebook.

Why this matters:

- every later cell reuses the same model state and prompt-local caches
- the feature sections load saved SAE checkpoints instead of retraining in the notebook
- failure is explicit: if an artifact is missing, the notebook stops instead of hiding the error


In [1]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

if Path.cwd().name == "notebook":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.kv_retrieve_analysis import (
    apply_rope,
    build_attention_tables,
    build_head_role_summary_table,
    build_head_source_contribution_table,
    build_head_source_write_table,
    build_kv_algorithm_variable_table,
    build_kv_prompt_layout_table,
    build_layer_feature_readout_table,
    build_mlp_neuron_batch_ablation_table,
    build_mlp_neuron_clean_corrupt_head_patch_table,
    build_mlp_neuron_clean_corrupt_source_patch_table,
    build_mlp_neuron_contribution_table,
    build_mlp_neuron_group_summary_table,
    build_mlp_neuron_read_comparison_table,
    build_mlp_neuron_upstream_head_effect_table,
    build_top_mlp_neuron_examples,
    build_ov_topk_table,
    build_path_patched_attention_table,
    build_qk_table,
    build_qkv_patched_attention_table,
    build_query_swap_head_role_comparison_table,
    build_query_swap_slot_routing_comparison_table,
    build_single_prompt_head_role_table,
    build_single_prompt_slot_routing_table,
    build_slot_routing_summary_table,
    build_stage_variable_readout_table,
    build_stage_variable_summary_table,
    collect_head_role_attention_table,
    collect_mlp_neuron_activation_table,
    collect_slot_routing_table,
    collect_stage_variable_readout_table,
    compute_path_patched_head_details,
    head_residual_contribution,
    load_checkpoint_model,
    load_dataset_bundle,
    make_query_swap_prompt,
    make_query_swap_row,
    ov_source_logits,
    residual_vector_to_logits,
    run_prompt,
    score_mlp_neuron_ablation_prompt,
    score_patched_prompt,
    score_path_patched_prompt,
    score_qkv_patched_prompt,
    score_rows_with_optional_ablation,
)
from scripts.kv_retrieve_features import (
    build_feature_activation_table,
    build_feature_encoder_contribution_table,
    build_feature_group_summary_table,
    collect_split_activations,
    intervene_on_sae_features,
    load_sae_checkpoint,
    score_feature_intervention,
    score_query_feature_intervention,
    select_feature_panel,
)

DEVICE = torch.device("cpu")
DATASET_DIR = PROJECT_ROOT / "dataset" / "kv_retrieve_3"
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "kv_retrieve_3" / "selected_checkpoint.pt"
L1H0_SAE_PATH = PROJECT_ROOT / "models" / "kv_retrieve_3" / "sae_block1_final_l1h0_l1e1.pt"
L2Q_SAE_PATH = PROJECT_ROOT / "models" / "kv_retrieve_3" / "sae_l2h0_final_q_l1e1.pt"
OUTPUT_DIR = PROJECT_ROOT / "notebook" / "outputs"
L1H0_SUMMARY_PATH = OUTPUT_DIR / "kv_retrieve_feature_basis_l1h0_site.json"
RESID_SUMMARY_PATH = OUTPUT_DIR / "kv_retrieve_feature_basis_l1e1.json"
L2Q_SUMMARY_PATH = OUTPUT_DIR / "kv_retrieve_feature_basis_l2h0_q.json"

for required_path in [
    DATASET_DIR,
    CHECKPOINT_PATH,
    L1H0_SAE_PATH,
    L2Q_SAE_PATH,
    L1H0_SUMMARY_PATH,
    RESID_SUMMARY_PATH,
    L2Q_SUMMARY_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required artifact: {required_path}")

bundle = load_dataset_bundle(DATASET_DIR)
checkpoint, analysis_model = load_checkpoint_model(CHECKPOINT_PATH, device=DEVICE)
_, feature_sae = load_sae_checkpoint(L1H0_SAE_PATH, device=DEVICE)
_, l2q_sae = load_sae_checkpoint(L2Q_SAE_PATH, device=DEVICE)

with L1H0_SUMMARY_PATH.open() as handle:
    l1h0_data = json.load(handle)
with RESID_SUMMARY_PATH.open() as handle:
    resid_data = json.load(handle)
with L2Q_SUMMARY_PATH.open() as handle:
    l2q_data = json.load(handle)

feature_df = pd.DataFrame(l1h0_data["feature_summary_top10"])
resid_feature_df = pd.DataFrame(resid_data["feature_summary_top10"])
l2q_feature_df = pd.DataFrame(l2q_data["feature_summary_top10"])

EXAMPLE_SPLIT = "test"
EXAMPLE_INDEX = 0
BATCH_LIMIT = 64
BATCH_PAIR_LIMIT = 64
NEURON_BATCH_LIMIT = 64

example = bundle.raw_splits[EXAMPLE_SPLIT][EXAMPLE_INDEX]
clean_prompt = example["prompt"]
clean_target = example["target"]
corrupt_prompt, corrupt_target = make_query_swap_prompt(example)
corrupt_row = make_query_swap_row(example)

clean_result, clean_cache = run_prompt(
    analysis_model,
    bundle,
    clean_prompt,
    DEVICE,
    expected_target=clean_target,
    return_cache=True,
)
corrupt_result, corrupt_cache = run_prompt(
    analysis_model,
    bundle,
    corrupt_prompt,
    DEVICE,
    expected_target=corrupt_target,
    return_cache=True,
)

if clean_cache is None or corrupt_cache is None:
    raise ValueError("Expected clean and corrupt caches in the unified notebook")

prompt_tokens = clean_prompt.split()
final_position_index = len(prompt_tokens) - 1
query_position_index = len(prompt_tokens) - 2
foil_token = clean_result["foil_token"]

target_positions = [index for index, token in enumerate(prompt_tokens) if token == clean_target]
if len(target_positions) != 1:
    raise ValueError(f"Expected exactly one clean target position, found {target_positions}")
clean_value_position = target_positions[0]

corrupt_value_positions = [index for index, token in enumerate(prompt_tokens) if token == corrupt_target]
if len(corrupt_value_positions) != 1:
    raise ValueError(f"Expected exactly one corrupt target position, found {corrupt_value_positions}")
corrupt_value_position = corrupt_value_positions[0]

clean_l1h0_source = head_residual_contribution(analysis_model, clean_cache, layer_index=0, head_index=0)
corrupt_l1h0_source = head_residual_contribution(analysis_model, corrupt_cache, layer_index=0, head_index=0)
clean_l1h0_vector = clean_l1h0_source[0, final_position_index].detach().cpu()
corrupt_l1h0_vector = corrupt_l1h0_source[0, final_position_index].detach().cpu()
clean_l2h0_q = clean_cache["blocks"][1]["attention"]["q"][0, 0, final_position_index, :].detach().cpu()
corrupt_l2h0_q = corrupt_cache["blocks"][1]["attention"]["q"][0, 0, final_position_index, :].detach().cpu()
clean_l2h0_pattern = clean_cache["blocks"][1]["attention"]["pattern"][0, 0, final_position_index, :].detach().cpu()
corrupt_l2h0_pattern = corrupt_cache["blocks"][1]["attention"]["pattern"][0, 0, final_position_index, :].detach().cpu()

available_example_feature_ids = [int(key) for key in l1h0_data["top_examples"].keys()]
feature_panel = select_feature_panel(
    feature_df,
    available_example_feature_ids=available_example_feature_ids,
    support_count=2,
    control_count=1,
)
support_features = feature_panel["support_features"]
control_features = feature_panel["control_features"]
selected_features = feature_panel["panel_features"]
selected_feature_roles = {feature_index: "support" for feature_index in support_features}
selected_feature_roles.update({feature_index: "control" for feature_index in control_features})

l2q_panel = select_feature_panel(
    l2q_feature_df,
    support_count=2,
    control_count=1,
)
downstream_feature_set = l2q_panel["support_features"]

display(pd.DataFrame([
    {
        "clean_prompt": clean_prompt,
        "clean_target": clean_target,
        "clean_prediction": clean_result["predicted_token"],
        "clean_margin": clean_result["margin"],
        "corrupt_prompt": corrupt_prompt,
        "corrupt_target": corrupt_target,
        "corrupt_prediction": corrupt_result["predicted_token"],
        "corrupt_margin": corrupt_result["margin"],
    }
]))


,clean_prompt,clean_target,clean_prediction,clean_margin,corrupt_prompt,corrupt_target,corrupt_prediction,corrupt_margin
0,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->,V0,V0,22.036973,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K7 ->,V2,V2,17.406853


## Prompt Layout And Single-Prompt Inspection

This cell fixes the anchor prompt in symbolic form and shows the raw final-position attention patterns.

Why this helps:

- it grounds the rest of the notebook in one concrete prompt
- it reveals the first query-head and value-head candidates without any patching
- it gives the token-level view before we move to features, neurons, and registers


In [2]:
variable_df = build_kv_algorithm_variable_table(example)
layout_df = build_kv_prompt_layout_table(example)

display(variable_df)
display(layout_df)

for title, attention_df in build_attention_tables(clean_prompt, clean_cache):
    print(title)
    display(attention_df)

head_focus_rows = []
for layer_index, block_cache in enumerate(clean_cache["blocks"]):
    pattern = block_cache["attention"]["pattern"][0, :, -1, :]
    for head_index in range(pattern.shape[0]):
        head_focus_rows.append(
            {
                "component": f"L{layer_index + 1}H{head_index}",
                "layer_index": layer_index,
                "head_index": head_index,
                "attention_to_query": float(pattern[head_index, query_position_index].item()),
                "attention_to_target_value": float(pattern[head_index, clean_value_position].item()),
            }
        )

head_focus_df = pd.DataFrame(head_focus_rows)
display(head_focus_df.sort_values(["attention_to_query", "attention_to_target_value"], ascending=False).reset_index(drop=True))

query_head_row = head_focus_df.sort_values("attention_to_query", ascending=False).iloc[0]
value_head_row = head_focus_df.sort_values("attention_to_target_value", ascending=False).iloc[0]

display(
    pd.DataFrame(
        [
            {
                "query_head_candidate": query_head_row["component"],
                "query_attention": query_head_row["attention_to_query"],
                "value_head_candidate": value_head_row["component"],
                "value_attention": value_head_row["attention_to_target_value"],
            }
        ]
    )
)


,prompt,query_key,query_key_position,selected_pair_index,selected_key_position,selected_value_position,target_value,distractor_key_positions,distractor_value_positions,distractor_keys,distractor_values
0,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->,K4,11,1,4,5,V0,"[1, 7]","[2, 8]","[K7, K6]","[V2, V7]"


,position,token,structural_role,algorithm_role,pair_index,matches_query_key,is_selected_pair
0,0,<bos>,bos,bos,NaN,False,False
1,1,K7,context_key,distractor_key,0.0,False,False
2,2,V2,context_value,distractor_value,0.0,False,False
3,3,;,separator,separator,0.0,False,False
4,4,K4,context_key,selected_key,1.0,True,True
5,5,V0,context_value,selected_value,1.0,True,True
6,6,;,separator,separator,1.0,False,True
7,7,K6,context_key,distractor_key,2.0,False,False
8,8,V7,context_value,distractor_value,2.0,False,False
9,9,;,separator,separator,2.0,False,False


Final-position attention, layer 1


,L1H0,L1H1
<bos>,0.003431,0.008855
K7,0.003693,0.091720
V2,0.001326,0.096897
;,0.042843,0.014092
K4,0.035790,0.087123
V0,0.073334,0.219449
;,0.051017,0.012214
K6,0.058149,0.172984
V7,0.640118,0.193592
;,0.058765,0.020630


Final-position attention, layer 2


,L2H0,L2H1
<bos>,0.000564,0.001942
K7,0.000699,0.042932
V2,0.000361,0.003872
;,0.001293,0.000159
K4,0.000393,0.000524
V0,0.725602,0.109736
;,0.098556,0.002019
K6,0.006205,0.000611
V7,0.000565,0.003076
;,0.005300,0.000003


,component,layer_index,head_index,attention_to_query,attention_to_target_value
0,L2H1,1,1,0.834365,0.109736
1,L1H1,0,1,0.020360,0.219449
2,L1H0,0,0,0.005567,0.073334
3,L2H0,1,0,0.001103,0.725602


,query_head_candidate,query_attention,value_head_candidate,value_attention
0,L2H1,0.834365,L2H0,0.725602


## Clean / Corrupt Query Swap

This is the simplest sanity check: change only the query key while leaving the context pairs fixed.

Why this helps:

- it shows whether the model is actually doing keyed retrieval instead of memorizing values
- it gives the clean/corrupt pair used by most later patching and tracing cells


In [3]:
query_swap_df = pd.DataFrame(
    [
        {"case": "clean", "prompt": clean_prompt, **clean_result},
        {"case": "corrupt_query_swap", "prompt": corrupt_prompt, **corrupt_result},
    ]
)[
    [
        "case",
        "prompt",
        "predicted_token",
        "target_token",
        "foil_token",
        "target_logit",
        "foil_logit",
        "margin",
        "correct",
    ]
]

display(query_swap_df)


,case,prompt,predicted_token,target_token,foil_token,target_logit,foil_logit,margin,correct
0,clean,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->,V0,V0,V2,29.739086,7.702113,22.036973,True
1,corrupt_query_swap,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K7 ->,V2,V2,V0,21.894787,4.487934,17.406853,True


## Activation Patching

This cell asks a broad causal question: if a clean activation is restored inside the corrupt run, does the clean answer come back?

Why this helps:

- it identifies which whole-block or whole-head activations are strong enough to rescue the behavior
- it is a coarse localization step before path-specific patching


In [4]:
patch_specs = [
    {"label": "clean baseline", "mode": "clean"},
    {"label": "corrupt no patch", "mode": "corrupt", "patch": None},
    {"label": "patch resid after block 1", "mode": "patch", "patch": {"kind": "resid_after_block", "layer_index": 0}},
    {"label": "patch resid after block 2", "mode": "patch", "patch": {"kind": "resid_after_block", "layer_index": 1}},
    {"label": "patch L1H0 head_out", "mode": "patch", "patch": {"kind": "head_out", "layer_index": 0, "head_index": 0}},
    {"label": "patch L1H1 head_out", "mode": "patch", "patch": {"kind": "head_out", "layer_index": 0, "head_index": 1}},
    {"label": "patch L2H0 head_out", "mode": "patch", "patch": {"kind": "head_out", "layer_index": 1, "head_index": 0}},
    {"label": "patch L2H1 head_out", "mode": "patch", "patch": {"kind": "head_out", "layer_index": 1, "head_index": 1}},
    {"label": "patch block 1 mlp_out", "mode": "patch", "patch": {"kind": "mlp_out", "layer_index": 0}},
    {"label": "patch block 2 mlp_out", "mode": "patch", "patch": {"kind": "mlp_out", "layer_index": 1}},
]

patch_rows = []
for spec in patch_specs:
    if spec["mode"] == "clean":
        result, _ = run_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            DEVICE,
            expected_target=clean_target,
            return_cache=False,
        )
    elif spec["mode"] == "corrupt":
        result = score_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            patch=None,
            clean_cache=clean_cache,
        )
    else:
        result = score_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            patch=spec["patch"],
            clean_cache=clean_cache,
        )

    patch_rows.append(
        {
            "case": spec["label"],
            "predicted_token": result["predicted_token"],
            "target_token": result["target_token"],
            "foil_token": result["foil_token"],
            "margin": result["margin"],
            "restores_clean_answer": result["predicted_token"] == clean_target,
        }
    )

patch_df = pd.DataFrame(patch_rows).sort_values("margin", ascending=False).reset_index(drop=True)
display(patch_df)


,case,predicted_token,target_token,foil_token,margin,restores_clean_answer
0,clean baseline,V0,V0,V2,22.036973,True
1,patch resid after block 2,V0,V0,V2,22.036973,True
2,patch L1H0 head_out,V0,V0,V2,17.144692,True
3,patch resid after block 1,V0,V0,V2,17.112144,True
4,patch L2H0 head_out,V0,V0,->,15.198623,True
5,patch L2H1 head_out,V2,V0,V2,-9.826786,False
6,patch block 1 mlp_out,V2,V0,V2,-13.729438,False
7,patch block 2 mlp_out,V2,V0,V2,-14.839757,False
8,corrupt no patch,V2,V0,V2,-17.406853,False
9,patch L1H1 head_out,V2,V0,V2,-17.529464,False


## Path Patching And Position-Resolved Path Patching

This section narrows the question from whole activations to specific source-to-destination paths.

Why this helps:

- it asks which upstream component matters specifically for the downstream value head
- the position-resolved view turns a broad head-to-head path into a token-local path
- this is the main bridge from coarse patching into a concrete retrieval circuit


In [5]:
if int(value_head_row["layer_index"]) == 0:
    raise RuntimeError("Path patching requires a destination head above layer 1")

destination = {
    "layer_index": int(value_head_row["layer_index"]),
    "head_index": int(value_head_row["head_index"]),
}
source_layer_index = destination["layer_index"] - 1
source_layer_label = source_layer_index + 1

path_specs = [
    {"label": "clean baseline", "mode": "clean"},
    {"label": "corrupt no patch", "mode": "corrupt"},
    {
        "label": f"path block {source_layer_label} residual -> {value_head_row['component']}",
        "mode": "path",
        "source_patch": {"kind": "resid_after_block", "layer_index": source_layer_index},
    },
]
for head_index in range(checkpoint["config"]["n_heads"]):
    path_specs.append(
        {
            "label": f"path L{source_layer_label}H{head_index} -> {value_head_row['component']}",
            "mode": "path",
            "source_patch": {
                "kind": "head_resid",
                "layer_index": source_layer_index,
                "head_index": head_index,
            },
        }
    )
path_specs.append(
    {
        "label": f"path block {source_layer_label} mlp_out -> {value_head_row['component']}",
        "mode": "path",
        "source_patch": {"kind": "mlp_out", "layer_index": source_layer_index},
    }
)

path_rows = []
for spec in path_specs:
    if spec["mode"] == "clean":
        result, _ = run_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            DEVICE,
            expected_target=clean_target,
            return_cache=False,
        )
    elif spec["mode"] == "corrupt":
        result = score_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            patch=None,
            clean_cache=clean_cache,
        )
    else:
        result = score_path_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            source_patch=spec["source_patch"],
            destination=destination,
            clean_cache=clean_cache,
            corrupt_cache=corrupt_cache,
        )

    path_rows.append(
        {
            "case": spec["label"],
            "predicted_token": result["predicted_token"],
            "foil_token": result["foil_token"],
            "margin": result["margin"],
            "restores_clean_answer": result["predicted_token"] == clean_target,
        }
    )

path_df = pd.DataFrame(path_rows).sort_values("margin", ascending=False).reset_index(drop=True)
display(path_df)

source_head_candidates = path_df[path_df["case"].str.contains(r"^path L\d+H\d+ ->", regex=True)].copy()
if source_head_candidates.empty:
    raise RuntimeError("No source-head path rows found")

source_head_candidates["source_head_index"] = (
    source_head_candidates["case"].str.extract(r"^path L\d+H(\d+) ->")[0].astype(int)
)
best_source_head_row = source_head_candidates.sort_values("margin", ascending=False).iloc[0]
best_source_head_index = int(best_source_head_row["source_head_index"])

position_path_rows = []
for source_position, source_token in enumerate(clean_prompt.split()):
    result = score_path_patched_prompt(
        analysis_model,
        bundle,
        clean_prompt,
        corrupt_prompt,
        clean_target,
        DEVICE,
        source_patch={
            "kind": "head_resid",
            "layer_index": source_layer_index,
            "head_index": best_source_head_index,
            "source_positions": [source_position],
        },
        destination=destination,
        clean_cache=clean_cache,
        corrupt_cache=corrupt_cache,
    )
    position_path_rows.append(
        {
            "source_position": source_position,
            "source_token": source_token,
            "predicted_token": result["predicted_token"],
            "foil_token": result["foil_token"],
            "margin": result["margin"],
            "restores_clean_answer": result["predicted_token"] == clean_target,
        }
    )

position_path_df = pd.DataFrame(position_path_rows).sort_values("margin", ascending=False).reset_index(drop=True)
display(position_path_df)


,case,predicted_token,foil_token,margin,restores_clean_answer
0,clean baseline,V0,V2,22.036973,True
1,path block 1 residual -> L2H0,V0,->,15.198623,True
2,path L1H0 -> L2H0,V0,V7,9.158729,True
3,path block 1 mlp_out -> L2H0,V2,V2,-17.291244,False
4,corrupt no patch,V2,V2,-17.406853,False
5,path L1H1 -> L2H0,V2,V2,-17.454963,False


,source_position,source_token,predicted_token,foil_token,margin,restores_clean_answer
0,12,->,V0,V7,9.122712,True
1,11,K4,V2,V2,-17.401617,False
2,0,<bos>,V2,V2,-17.406853,False
3,1,K7,V2,V2,-17.406853,False
4,2,V2,V2,V2,-17.406853,False
5,3,;,V2,V2,-17.406853,False
6,4,K4,V2,V2,-17.406853,False
7,5,V0,V2,V2,-17.406853,False
8,6,;,V2,V2,-17.406853,False
9,7,K6,V2,V2,-17.406853,False


## Destination Routing And Q / K / V Tests

This cell asks two tighter questions:

- which downstream head actually receives the useful source signal?
- inside that head, is the critical channel `Q`, `K`, or `V`?

Why this helps:

- it turns a broad source-to-layer statement into a source-to-head statement
- it isolates whether the mechanism is query retargeting or value overwriting


In [6]:
destination_specific_rows = []
for destination_head_index in range(checkpoint["config"]["n_heads"]):
    destination_spec = {
        "layer_index": destination["layer_index"],
        "head_index": destination_head_index,
    }
    source_specs = [
        {
            "source_label": f"block {source_layer_label} residual at final position",
            "source_patch": {
                "kind": "resid_after_block",
                "layer_index": source_layer_index,
                "source_positions": [final_position_index],
            },
        },
        {
            "source_label": f"L{source_layer_label}H{best_source_head_index} at final position",
            "source_patch": {
                "kind": "head_resid",
                "layer_index": source_layer_index,
                "head_index": best_source_head_index,
                "source_positions": [final_position_index],
            },
        },
    ]

    for spec in source_specs:
        result = score_path_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            source_patch=spec["source_patch"],
            destination=destination_spec,
            clean_cache=clean_cache,
            corrupt_cache=corrupt_cache,
        )
        destination_specific_rows.append(
            {
                "source": spec["source_label"],
                "destination": f"L{destination_spec['layer_index'] + 1}H{destination_spec['head_index']}",
                "predicted_token": result["predicted_token"],
                "foil_token": result["foil_token"],
                "margin": result["margin"],
                "restores_clean_answer": result["predicted_token"] == clean_target,
            }
        )

final_position_destination_df = (
    pd.DataFrame(destination_specific_rows)
    .sort_values(["source", "margin"], ascending=[True, False])
    .reset_index(drop=True)
)
display(final_position_destination_df)

qkv_specs = [
    {"label": "clean baseline", "mode": "clean"},
    {"label": "corrupt no patch", "mode": "corrupt"},
    {"label": "patch Q only", "mode": "qkv", "components": ["q"]},
    {"label": "patch K only", "mode": "qkv", "components": ["k"]},
    {"label": "patch V only", "mode": "qkv", "components": ["v"]},
    {"label": "patch QK", "mode": "qkv", "components": ["q", "k"]},
    {"label": "patch QV", "mode": "qkv", "components": ["q", "v"]},
    {"label": "patch KV", "mode": "qkv", "components": ["k", "v"]},
    {"label": "patch QKV", "mode": "qkv", "components": ["q", "k", "v"]},
]

qkv_rows = []
for spec in qkv_specs:
    if spec["mode"] == "clean":
        result, _ = run_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            DEVICE,
            expected_target=clean_target,
            return_cache=False,
        )
    elif spec["mode"] == "corrupt":
        result = score_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            patch=None,
            clean_cache=clean_cache,
        )
    else:
        result = score_qkv_patched_prompt(
            analysis_model,
            bundle,
            clean_prompt,
            corrupt_prompt,
            clean_target,
            DEVICE,
            destination=destination,
            components=spec["components"],
            clean_cache=clean_cache,
            corrupt_cache=corrupt_cache,
        )

    qkv_rows.append(
        {
            "case": spec["label"],
            "predicted_token": result["predicted_token"],
            "foil_token": result["foil_token"],
            "margin": result["margin"],
            "restores_clean_answer": result["predicted_token"] == clean_target,
        }
    )

qkv_df = pd.DataFrame(qkv_rows).sort_values("margin", ascending=False).reset_index(drop=True)
display(qkv_df)


,source,destination,predicted_token,foil_token,margin,restores_clean_answer
0,L1H0 at final position,L2H0,V0,V7,9.122712,True
1,L1H0 at final position,L2H1,V2,V2,-16.214856,False
2,block 1 residual at final position,L2H0,V0,->,15.200890,True
3,block 1 residual at final position,L2H1,V2,V2,-15.051017,False


,case,predicted_token,foil_token,margin,restores_clean_answer
0,clean baseline,V0,V2,22.036973,True
1,patch QK,V0,->,15.201979,True
2,patch QKV,V0,->,15.200890,True
3,patch Q only,V0,->,15.165829,True
4,patch QV,V0,->,15.081021,True
5,patch KV,V2,V2,-17.358109,False
6,patch K only,V2,V2,-17.377025,False
7,patch V only,V2,V2,-17.404651,False
8,corrupt no patch,V2,V2,-17.406853,False


## Batch Causal Checks

The previous cells are prompt-local. This cell checks whether the same causal story survives across a batch.

Why this helps:

- it prevents overfitting to the anchor prompt
- it keeps both kinds of evidence in view:
  - path-specific restoration on clean/corrupt pairs
  - direct head ablations on baseline-correct prompts


In [7]:
batch_pairs = []
for row in bundle.raw_splits["test"]:
    clean_result_batch, clean_cache_batch = run_prompt(
        analysis_model,
        bundle,
        row["prompt"],
        DEVICE,
        expected_target=row["target"],
        return_cache=True,
    )
    corrupt_prompt_batch, corrupt_target_batch = make_query_swap_prompt(row)
    corrupt_result_batch, corrupt_cache_batch = run_prompt(
        analysis_model,
        bundle,
        corrupt_prompt_batch,
        DEVICE,
        expected_target=corrupt_target_batch,
        return_cache=True,
    )
    if clean_result_batch["predicted_token"] != row["target"]:
        continue
    batch_pairs.append(
        {
            "clean_prompt": row["prompt"],
            "clean_target": row["target"],
            "corrupt_prompt": corrupt_prompt_batch,
            "corrupt_target": corrupt_target_batch,
            "clean_cache": clean_cache_batch,
            "corrupt_cache": corrupt_cache_batch,
        }
    )

if len(batch_pairs) < BATCH_PAIR_LIMIT:
    raise RuntimeError(
        f"Expected at least {BATCH_PAIR_LIMIT} clean/corrupt test pairs, found {len(batch_pairs)}"
    )
batch_pairs = batch_pairs[:BATCH_PAIR_LIMIT]

batch_case_specs = [
    {"label": "corrupt no patch", "kind": "corrupt"},
    {
        "label": "patch L1H0 final-pos -> destination",
        "kind": "path",
        "source_patch": {"kind": "head_resid", "layer_index": 0, "head_index": 0, "source_positions": [final_position_index]},
        "destination": destination,
    },
    {
        "label": "patch Q into destination",
        "kind": "qkv",
        "destination": destination,
        "components": ["q"],
    },
    {
        "label": "patch V into destination",
        "kind": "qkv",
        "destination": destination,
        "components": ["v"],
    },
]

batch_rows = []
for spec in batch_case_specs:
    restore_flags = []
    margins = []
    for pair in batch_pairs:
        if spec["kind"] == "corrupt":
            result = score_patched_prompt(
                analysis_model,
                bundle,
                pair["clean_prompt"],
                pair["corrupt_prompt"],
                pair["clean_target"],
                DEVICE,
                patch=None,
                clean_cache=pair["clean_cache"],
            )
        elif spec["kind"] == "path":
            result = score_path_patched_prompt(
                analysis_model,
                bundle,
                pair["clean_prompt"],
                pair["corrupt_prompt"],
                pair["clean_target"],
                DEVICE,
                source_patch=spec["source_patch"],
                destination=spec["destination"],
                clean_cache=pair["clean_cache"],
                corrupt_cache=pair["corrupt_cache"],
            )
        else:
            result = score_qkv_patched_prompt(
                analysis_model,
                bundle,
                pair["clean_prompt"],
                pair["corrupt_prompt"],
                pair["clean_target"],
                DEVICE,
                destination=spec["destination"],
                components=spec["components"],
                clean_cache=pair["clean_cache"],
                corrupt_cache=pair["corrupt_cache"],
            )
        restore_flags.append(result["predicted_token"] == pair["clean_target"])
        margins.append(result["margin"])

    batch_rows.append(
        {
            "case": spec["label"],
            "num_pairs": len(batch_pairs),
            "restore_rate": sum(restore_flags) / len(restore_flags),
            "mean_margin_vs_clean_target": float(sum(margins) / len(margins)),
        }
    )

batch_path_summary_df = pd.DataFrame(batch_rows).sort_values("mean_margin_vs_clean_target", ascending=False).reset_index(drop=True)
display(batch_path_summary_df)

baseline_test_df = score_rows_with_optional_ablation(
    analysis_model,
    bundle,
    bundle.raw_splits["test"][:BATCH_LIMIT],
    DEVICE,
)
baseline_correct_mask = baseline_test_df["correct"].tolist()
baseline_correct_rows = [
    row for row, is_correct in zip(bundle.raw_splits["test"][:BATCH_LIMIT], baseline_correct_mask) if is_correct
]
baseline_correct_df = baseline_test_df.loc[baseline_test_df["correct"]].reset_index(drop=True)
if baseline_correct_df.empty:
    raise RuntimeError("Baseline model has no correct prompts in the selected batch")

batch_ablation_summary_rows = [
    {
        "component": "baseline on baseline-correct subset",
        "num_prompts": len(baseline_correct_df),
        "accuracy": baseline_correct_df["correct"].mean(),
        "mean_margin": baseline_correct_df["margin"].mean(),
        "margin_delta_vs_baseline": 0.0,
        "prediction_change_rate": 0.0,
        "flip_rate_from_correct": 0.0,
    }
]

for layer_index in range(checkpoint["config"]["n_layers"]):
    for head_index in range(checkpoint["config"]["n_heads"]):
        ablated_df = score_rows_with_optional_ablation(
            analysis_model,
            bundle,
            baseline_correct_rows,
            DEVICE,
            ablation={"layer_index": layer_index, "head_index": head_index},
        )
        batch_ablation_summary_rows.append(
            {
                "component": f"L{layer_index + 1}H{head_index} final-position ablated",
                "num_prompts": len(ablated_df),
                "accuracy": ablated_df["correct"].mean(),
                "mean_margin": ablated_df["margin"].mean(),
                "margin_delta_vs_baseline": ablated_df["margin"].mean() - baseline_correct_df["margin"].mean(),
                "prediction_change_rate": (ablated_df["predicted_token"] != baseline_correct_df["predicted_token"]).mean(),
                "flip_rate_from_correct": (~ablated_df["correct"]).mean(),
            }
        )

batch_ablation_summary_df = pd.DataFrame(batch_ablation_summary_rows).sort_values("margin_delta_vs_baseline").reset_index(drop=True)
display(batch_ablation_summary_df)


,case,num_pairs,restore_rate,mean_margin_vs_clean_target
0,patch Q into destination,64,0.484375,-1.107974
1,patch L1H0 final-pos -> destination,64,0.359375,-4.679773
2,corrupt no patch,64,0.015625,-17.173402
3,patch V into destination,64,0.015625,-17.177153


,component,num_prompts,accuracy,mean_margin,margin_delta_vs_baseline,prediction_change_rate,flip_rate_from_correct
0,L1H0 final-position ablated,61,0.508197,-1.881690,-14.394481,0.491803,0.491803
1,L2H0 final-position ablated,61,0.737705,2.907902,-9.604888,0.262295,0.262295
2,L2H1 final-position ablated,61,0.704918,3.982462,-8.530329,0.295082,0.295082
3,L1H1 final-position ablated,61,0.918033,10.414634,-2.098156,0.081967,0.081967
4,baseline on baseline-correct subset,61,1.000000,12.512791,0.000000,0.000000,0.000000


## QK / OV Analysis

This cell exposes the two most direct attention-head objects:

- QK ranking: where a head wants to look
- OV write: what a head writes when it reads a source

Why this helps:

- it separates routing from content
- it makes the query-like head and value-like head legible in the model's own math


In [8]:
query_layer_index = int(query_head_row["layer_index"])
query_head_index = int(query_head_row["head_index"])
value_layer_index = int(value_head_row["layer_index"])
value_head_index = int(value_head_row["head_index"])

print(f"QK analysis head: L{query_layer_index + 1}H{query_head_index}")
clean_qk_df = build_qk_table(clean_prompt, clean_cache, query_layer_index, query_head_index)
corrupt_qk_df = build_qk_table(corrupt_prompt, corrupt_cache, query_layer_index, query_head_index)
display(clean_qk_df.sort_values("qk_score", ascending=False).reset_index(drop=True))
display(corrupt_qk_df.sort_values("qk_score", ascending=False).reset_index(drop=True))

print(f"OV analysis head: L{value_layer_index + 1}H{value_head_index}")
clean_target_id = bundle.token_to_id[clean_target]
foil_token_id = bundle.token_to_id[foil_token]
value_positions = [index for index, token in enumerate(prompt_tokens) if token in bundle.value_tokens]

ov_summary_rows = []
for source_position in value_positions:
    source_logits = ov_source_logits(
        analysis_model,
        clean_cache,
        value_layer_index,
        value_head_index,
        source_position,
    )
    top_token_id = int(source_logits.argmax().item())
    ov_summary_rows.append(
        {
            "source_position": source_position,
            "source_token": prompt_tokens[source_position],
            "top_written_token": bundle.id_to_token[top_token_id],
            "top_written_logit": float(source_logits[top_token_id].item()),
            "target_logit": float(source_logits[clean_target_id].item()),
            "foil_logit": float(source_logits[foil_token_id].item()),
            "target_minus_foil": float((source_logits[clean_target_id] - source_logits[foil_token_id]).item()),
        }
    )

ov_summary_df = pd.DataFrame(ov_summary_rows).sort_values("target_minus_foil", ascending=False).reset_index(drop=True)
display(ov_summary_df)
display(build_ov_topk_table(analysis_model, bundle, clean_cache, value_layer_index, value_head_index, clean_value_position))


QK analysis head: L2H1


,position,token,qk_score,attention_weight
0,11,K4,1.963078,0.834365
1,5,V0,-0.065511,0.109736
2,1,K7,-1.003967,0.042932
3,2,V2,-3.409869,0.003872
4,8,V7,-3.639951,0.003076
5,6,;,-4.060928,0.002019
6,0,<bos>,-4.099718,0.001942
7,7,K6,-5.256419,0.000611
8,10,Q,-5.278840,0.000597
9,4,K4,-5.410684,0.000524


,position,token,qk_score,attention_weight
0,1,K7,7.310372,0.547483
1,11,K7,6.841518,0.342570
2,12,->,4.993831,0.053989
3,9,;,4.877013,0.048037
4,8,V7,2.671663,0.005294
5,10,Q,1.060254,0.001057
6,7,K6,0.548241,0.000633
7,4,K4,0.114999,0.000411
8,3,;,-0.393352,0.000247
9,0,<bos>,-0.699269,0.000182


OV analysis head: L2H0


,source_position,source_token,top_written_token,top_written_logit,target_logit,foil_logit,target_minus_foil
0,5,V0,V0,47.443581,47.443581,13.656315,33.787266
1,8,V7,V3,20.933960,8.678171,-4.399082,13.077253
2,2,V2,V2,46.925892,-1.973641,46.925892,-48.899532


,token,logit
0,V0,47.443581
1,V2,13.656315
2,V1,11.314699
3,K7,9.101694
4,;,8.353251


## Circuit Trace: `L1H0 -> Destination.Q -> Destination Write`

This is the compact computational trace section.

Why this helps:

- it checks whether the upstream write predicts the downstream query shift
- it compares a path patch against a direct Q patch
- it decomposes the destination head's final write by source position


In [9]:
l1h0_final_source_patch = {
    "kind": "head_resid",
    "layer_index": 0,
    "head_index": 0,
    "source_positions": [final_position_index],
}

clean_destination_qk_df = build_qk_table(clean_prompt, clean_cache, layer_index=destination["layer_index"], head_index=destination["head_index"]).rename(
    columns={"position": "source_position", "token": "source_token"}
)
corrupt_destination_qk_df = build_qk_table(corrupt_prompt, corrupt_cache, layer_index=destination["layer_index"], head_index=destination["head_index"]).rename(
    columns={"position": "source_position", "token": "source_token"}
)
path_destination_qk_df = build_path_patched_attention_table(
    analysis_model,
    corrupt_prompt,
    clean_cache,
    corrupt_cache,
    source_patch=l1h0_final_source_patch,
    destination=destination,
)
q_patch_destination_qk_df = build_qkv_patched_attention_table(
    analysis_model,
    corrupt_prompt,
    clean_cache,
    corrupt_cache,
    destination=destination,
    components=["q"],
)

score_retarget_df = clean_destination_qk_df.rename(
    columns={"source_token": "clean_source_token", "qk_score": "clean_qk_score", "attention_weight": "clean_attention"}
)
score_retarget_df = score_retarget_df.merge(
    corrupt_destination_qk_df.rename(
        columns={"source_token": "corrupt_source_token", "qk_score": "corrupt_qk_score", "attention_weight": "corrupt_attention"}
    ),
    on=["source_position"],
)
score_retarget_df = score_retarget_df.merge(
    path_destination_qk_df.rename(
        columns={"source_token": "path_source_token", "qk_score": "path_qk_score", "attention_weight": "path_attention"}
    ),
    on=["source_position"],
)
score_retarget_df = score_retarget_df.merge(
    q_patch_destination_qk_df.rename(
        columns={"source_token": "q_patch_source_token", "qk_score": "q_patch_qk_score", "attention_weight": "q_patch_attention"}
    ),
    on=["source_position"],
)
score_retarget_df["path_qk_delta_vs_corrupt"] = score_retarget_df["path_qk_score"] - score_retarget_df["corrupt_qk_score"]
score_retarget_df["q_patch_qk_delta_vs_corrupt"] = score_retarget_df["q_patch_qk_score"] - score_retarget_df["corrupt_qk_score"]
score_retarget_df["path_attention_delta_vs_corrupt"] = score_retarget_df["path_attention"] - score_retarget_df["corrupt_attention"]
score_retarget_df["q_patch_attention_delta_vs_corrupt"] = score_retarget_df["q_patch_attention"] - score_retarget_df["corrupt_attention"]

path_details = compute_path_patched_head_details(
    analysis_model,
    clean_cache,
    corrupt_cache,
    source_patch=l1h0_final_source_patch,
    destination=destination,
)

clean_l1h0_write = head_residual_contribution(analysis_model, clean_cache, layer_index=0, head_index=0)[0, final_position_index].detach().cpu()
corrupt_l1h0_write = head_residual_contribution(analysis_model, corrupt_cache, layer_index=0, head_index=0)[0, final_position_index].detach().cpu()
delta_h = clean_l1h0_write - corrupt_l1h0_write

destination_attn = analysis_model.blocks[destination["layer_index"]].attn
head_dim = destination_attn.head_dim
head_start = destination["head_index"] * head_dim
head_stop = head_start + head_dim
w_q_head = destination_attn.q_proj.weight[head_start:head_stop, :].detach().cpu()

linear_delta_q = torch.matmul(w_q_head, delta_h)
cos_final = destination_attn.rope_cos[final_position_index].detach().cpu().view(1, 1, 1, -1)
sin_final = destination_attn.rope_sin[final_position_index].detach().cpu().view(1, 1, 1, -1)
predicted_delta_q = apply_rope(linear_delta_q.view(1, 1, 1, -1), cos_final, sin_final).view(-1)

corrupt_q = corrupt_cache["blocks"][destination["layer_index"]]["attention"]["q"][0, destination["head_index"], final_position_index, :].detach().cpu()
path_q = path_details["q"][0, destination["head_index"], final_position_index, :].detach().cpu()
actual_delta_q = path_q - corrupt_q
corrupt_k = corrupt_cache["blocks"][destination["layer_index"]]["attention"]["k"][0, destination["head_index"], :, :].detach().cpu()

query_write_summary_df = pd.DataFrame(
    [
        {
            "clean_L1H0_final_norm": float(clean_l1h0_write.norm().item()),
            "corrupt_L1H0_final_norm": float(corrupt_l1h0_write.norm().item()),
            "delta_write_norm": float(delta_h.norm().item()),
            "predicted_delta_q_norm": float(predicted_delta_q.norm().item()),
            "actual_delta_q_norm": float(actual_delta_q.norm().item()),
            "delta_q_cosine_similarity": float(F.cosine_similarity(predicted_delta_q.unsqueeze(0), actual_delta_q.unsqueeze(0)).item()),
        }
    ]
)

query_effect_rows = []
for source_position, source_token in enumerate(clean_prompt.split()):
    approx_score_delta = float(torch.dot(predicted_delta_q, corrupt_k[source_position]) / math.sqrt(head_dim))
    actual_score_delta = float(
        score_retarget_df.loc[
            score_retarget_df["source_position"] == source_position,
            "path_qk_delta_vs_corrupt",
        ].iloc[0]
    )
    query_effect_rows.append(
        {
            "source_position": source_position,
            "source_token": source_token,
            "approx_score_delta_from_L1H0_write": approx_score_delta,
            "actual_score_delta_from_path_patch": actual_score_delta,
        }
    )
query_effect_df = pd.DataFrame(query_effect_rows).sort_values("actual_score_delta_from_path_patch", ascending=False).reset_index(drop=True)

destination_source_write_df = build_head_source_write_table(
    model=analysis_model,
    bundle=bundle,
    prompt=clean_prompt,
    attention_pattern=clean_cache["blocks"][destination["layer_index"]]["attention"]["pattern"][0, destination["head_index"], final_position_index, :].detach().cpu(),
    value_vectors=clean_cache["blocks"][destination["layer_index"]]["attention"]["v"][0, destination["head_index"], :, :].detach().cpu(),
    layer_index=destination["layer_index"],
    head_index=destination["head_index"],
    target_token=clean_target,
    foil_token=foil_token,
)

display(query_write_summary_df)
display(score_retarget_df.sort_values("path_qk_delta_vs_corrupt", ascending=False).reset_index(drop=True))
display(query_effect_df)
display(destination_source_write_df.sort_values("target_minus_foil", ascending=False).reset_index(drop=True))


,clean_L1H0_final_norm,corrupt_L1H0_final_norm,delta_write_norm,predicted_delta_q_norm,actual_delta_q_norm,delta_q_cosine_similarity
0,2.998705,8.62947,9.424322,12.716252,7.365511,0.922423


,source_position,clean_source_token,clean_qk_score,clean_attention,corrupt_source_token,corrupt_qk_score,corrupt_attention,path_source_token,path_qk_score,path_attention,q_patch_source_token,q_patch_qk_score,q_patch_attention,path_qk_delta_vs_corrupt,q_patch_qk_delta_vs_corrupt,path_attention_delta_vs_corrupt,q_patch_attention_delta_vs_corrupt
0,10,Q,2.414658,0.159159,Q,-0.670277,0.000184,Q,3.721483,0.329718,Q,2.414658,0.156810,4.391760,3.084935,0.329535,0.156627
1,5,V0,3.931754,0.725602,V0,0.078595,0.000388,V0,4.121330,0.491807,V0,3.931754,0.714893,4.042735,3.853159,0.491418,0.714504
2,12,->,-4.268593,0.000199,->,0.056582,0.000380,->,2.071730,0.063338,->,0.097715,0.015457,2.015148,0.041133,0.062958,0.015078
3,6,;,1.935379,0.098556,;,0.491970,0.000587,;,1.778679,0.047249,;,1.935379,0.097102,1.286709,1.443409,0.046662,0.096514
4,7,K6,-0.829819,0.006205,K6,1.251952,0.001255,K6,0.981785,0.021296,K6,-0.829819,0.006114,-0.270166,-2.081770,0.020041,0.004858
5,9,;,-0.987505,0.005300,;,1.307940,0.001328,;,0.511367,0.013305,;,-0.987505,0.005222,-0.796572,-2.295445,0.011977,0.003894
6,0,<bos>,-3.227711,0.000564,<bos>,-0.312452,0.000263,<bos>,-1.184798,0.002440,<bos>,-3.227711,0.000556,-0.872346,-2.915260,0.002177,0.000293
7,1,K7,-3.013061,0.000699,K7,3.227347,0.009051,K7,0.110515,0.008911,K7,-3.013061,0.000689,-3.116831,-6.240408,-0.000140,-0.008362
8,8,V7,-3.225801,0.000565,V7,1.004755,0.000981,V7,-2.159984,0.000920,V7,-3.225801,0.000557,-3.164739,-4.230556,-0.000060,-0.000424
9,11,K4,-2.557652,0.001103,K7,2.180800,0.003178,K7,-1.002095,0.002929,K7,-3.178183,0.000584,-3.182896,-5.358983,-0.000249,-0.002594


,source_position,source_token,approx_score_delta_from_L1H0_write,actual_score_delta_from_path_patch
0,10,Q,5.298078,4.391760
1,5,V0,4.254263,4.042735
2,12,->,1.214479,2.015148
3,6,;,0.924599,1.286709
4,7,K6,-1.483080,-0.270166
5,9,;,-2.284157,-0.796572
6,0,<bos>,-0.846458,-0.872346
7,1,K7,-6.421657,-3.116831
8,8,V7,-4.334518,-3.164739
9,11,K4,-5.255547,-3.182896


,position,token,attention_weight,value_norm,mix_component_norm,write_norm,target_logit,foil_logit,target_minus_foil
0,5,V0,0.725602,6.314266,4.581645,6.533440,34.425159,9.909048,24.516111
1,6,;,0.098556,5.385197,0.530744,0.851770,3.838442,-0.047636,3.886077
2,10,Q,0.159159,4.520470,0.719475,0.959098,0.733554,0.448254,0.285301
3,7,K6,0.006205,6.130124,0.038040,0.054129,0.057819,0.010972,0.046847
4,0,<bos>,0.000564,4.614021,0.002603,0.004408,0.006264,-0.002682,0.008945
5,1,K7,0.000699,6.007521,0.004200,0.006813,0.008424,0.000834,0.007590
6,8,V7,0.000565,4.465368,0.002524,0.003530,0.004905,-0.002486,0.007391
7,11,K4,0.001103,5.506171,0.006071,0.008477,0.022589,0.017291,0.005298
8,9,;,0.005300,4.082385,0.021637,0.029930,-0.036649,-0.039883,0.003233
9,12,->,0.000199,3.755977,0.000748,0.000957,0.003178,0.001507,0.001671


## Sparse Feature Setup And Site Comparison

This cell brings the SAE artifacts into the same notebook.

Why this helps:

- it compares the broad residual site against the narrowed `L1H0` site
- it also compares the upstream `L1H0` site against the downstream `L2H0.Q` site
- it selects one support/control feature panel dynamically instead of hardcoding feature ids


In [10]:
site_compare_df = pd.DataFrame(
    [
        {
            "site": resid_data["site"],
            "site_description": resid_data["site_description"],
            **resid_data["history_tail"][-1],
        },
        {
            "site": l1h0_data["site"],
            "site_description": l1h0_data["site_description"],
            **l1h0_data["history_tail"][-1],
        },
    ]
)
l2q_compare_df = pd.DataFrame(
    [
        {
            "site": l1h0_data["site"],
            "site_description": l1h0_data["site_description"],
            **l1h0_data["history_tail"][-1],
        },
        {
            "site": l2q_data["site"],
            "site_description": l2q_data["site_description"],
            **l2q_data["history_tail"][-1],
        },
    ]
)

display(site_compare_df[["site", "site_description", "val_recon_loss", "val_mean_active_features", "train_recon_loss", "train_mean_active_features"]])
display(l2q_compare_df[["site", "site_description", "val_recon_loss", "val_mean_active_features", "train_recon_loss", "train_mean_active_features"]])
display(feature_df[["feature_index", "mean_clean_activation", "mean_corrupt_activation", "mean_delta", "query_alignment", "query_cosine", "mechanism_score", "top_logit_tokens"]])
display(l2q_feature_df[["feature_index", "mean_clean_activation", "mean_corrupt_activation", "mean_delta", "query_alignment", "query_cosine", "mechanism_score", "projection_space"]])
display(pd.DataFrame([{
    "support_features": support_features,
    "control_features": control_features,
    "downstream_support_features": downstream_feature_set,
}]))


,site,site_description,val_recon_loss,val_mean_active_features,train_recon_loss,train_mean_active_features
0,block1_final_resid,Block 1 residual stream after MLP at the final position,0.008442,160.972000,0.007129,161.355225
1,block1_final_l1h0,L1H0 residual contribution at the final position,0.000483,96.129997,0.000482,96.169189


,site,site_description,val_recon_loss,val_mean_active_features,train_recon_loss,train_mean_active_features
0,block1_final_l1h0,L1H0 residual contribution at the final position,0.000483,96.129997,0.000482,96.169189
1,l2h0_final_q,L2H0 query vector at the final position,0.005558,84.533997,0.005137,84.591797


,feature_index,mean_clean_activation,mean_corrupt_activation,mean_delta,query_alignment,query_cosine,mechanism_score,top_logit_tokens
0,40,0.221635,0.085245,0.136390,0.535981,0.610666,0.073102,"[V5, Q, V0, V1, K0]"
1,30,0.366907,0.235702,0.131205,0.548841,0.573580,0.072011,"[K1, V3, K4, V5, K7]"
2,136,0.521745,0.655575,-0.133830,-0.442045,-0.511204,0.059159,"[<bos>, ;, K6, V2, K2]"
3,194,0.332289,0.442233,-0.109944,-0.471254,-0.516223,0.051811,"[K5, K7, ->, V7, <bos>]"
4,370,0.334113,0.420967,-0.086854,-0.560601,-0.680913,0.048691,"[<bos>, V6, K2, K3, K5]"
5,94,0.288533,0.184156,0.104376,0.405423,0.474043,0.042317,"[K4, K1, V0, V2, K6]"
6,289,0.191708,0.118593,0.073114,0.414933,0.579807,0.030338,"[V0, K1, K2, K3, K7]"
7,356,0.178469,0.113271,0.065198,0.412965,0.663043,0.026925,"[;, K7, V6, Q, K4]"
8,9,0.181477,0.263850,-0.082374,-0.283564,-0.329989,0.023358,"[K7, K2, K6, V3, K1]"
9,374,0.194638,0.138547,0.056091,0.391633,0.475389,0.021967,"[K4, ;, <bos>, V3, K3]"


,feature_index,mean_clean_activation,mean_corrupt_activation,mean_delta,query_alignment,query_cosine,mechanism_score,projection_space
0,5,0.330641,0.445538,-0.114898,-0.482153,-0.612842,0.055398,query_space
1,100,0.233932,0.307674,-0.073742,-0.384851,-0.489166,0.028380,query_space
2,40,0.187117,0.258694,-0.071577,-0.371680,-0.472424,0.026604,query_space
3,132,0.345789,0.417115,-0.071325,-0.354491,-0.450576,0.025284,query_space
4,21,0.347934,0.423798,-0.075864,-0.298557,-0.379482,0.022650,query_space
5,82,0.385214,0.481640,-0.096426,-0.227306,-0.288917,0.021918,query_space
6,93,0.134884,0.085345,0.049539,0.438595,0.557478,0.021728,query_space
7,37,0.136772,0.088444,0.048328,0.441537,0.561216,0.021339,query_space
8,177,0.207135,0.140471,0.066663,0.314910,0.400267,0.020993,query_space
9,166,0.326377,0.252403,0.073974,0.255700,0.325008,0.018915,query_space


,support_features,control_features,downstream_support_features
0,"[40, 30]",[370],"[93, 37]"


## Feature Dashboard And Label Semantics

This section turns the saved SAE features into prompt-level objects.

Why this helps:

- top examples show what a feature tends to fire on
- grouped label summaries test whether a feature is query-like, value-like, or position-like
- this is the feature-level analogue of the head focus tables above


In [11]:
def feature_examples_table(data: dict, feature_index: int) -> pd.DataFrame:
    key = str(feature_index)
    if key not in data["top_examples"]:
        raise KeyError(f"Feature {feature_index} not present in saved top_examples")
    return pd.DataFrame(data["top_examples"][key])

selected_feature_summary_df = (
    feature_df[feature_df["feature_index"].isin(selected_features)]
    .sort_values("mechanism_score", ascending=False)
    .reset_index(drop=True)
)
display(selected_feature_summary_df)

for feature_index in selected_features:
    display(Markdown(f"### Feature `{feature_index}` ({selected_feature_roles[feature_index]}) top examples"))
    display(feature_examples_table(l1h0_data, feature_index))

test_activations, test_records = collect_split_activations(
    analysis_model,
    bundle,
    split="test",
    site="block1_final_l1h0",
)
feature_activation_df = build_feature_activation_table(
    feature_sae,
    test_activations,
    test_records,
    bundle,
    selected_features,
)

preview_columns = [
    "split",
    "index",
    "query_key",
    "target",
    "query_pair_index",
    "correct_value_position",
    "corrupt_target",
] + [f"feature_{feature_index}_activation" for feature_index in selected_features]
display(feature_activation_df[preview_columns].head(8))

for group_column in ["query_key", "target", "correct_value_position", "query_pair_index", "corrupt_target"]:
    print(f"Grouped by {group_column}")
    display(build_feature_group_summary_table(feature_activation_df, selected_features, group_column))


,feature_index,mean_clean_activation,mean_corrupt_activation,mean_delta,abs_mean_delta,clean_activation_rate,decoder_norm,projected_q_norm,query_alignment,query_cosine,top_logit_tokens,top_logit_values,mechanism_score
0,40,0.221635,0.085245,0.136390,0.136390,0.429688,1.0,1.115602,0.535981,0.610666,"[V5, Q, V0, V1, K0]","[7.171279430389404, 5.237597942352295, 4.1218438148498535, 3.7430121898651123, 3.26385498046875]",0.073102
1,30,0.366907,0.235702,0.131205,0.131205,0.457031,1.0,1.216231,0.548841,0.573580,"[K1, V3, K4, V5, K7]","[9.63432502746582, 7.724791526794434, 6.822149276733398, 4.542654037475586, 4.404752731323242]",0.072011
2,370,0.334113,0.420967,-0.086854,0.086854,0.468750,1.0,1.046467,-0.560601,-0.680913,"[<bos>, V6, K2, K3, K5]","[7.488486289978027, 6.376976013183594, 3.0376853942871094, 1.4523577690124512, 0.20641374588012695]",0.048691


### Feature `40` (support) top examples

,activation,split,index,prompt,target,query_key
0,2.112178,test,253,<bos> K7 V1 ; K6 V3 ; K5 V4 ; Q K5 ->,V4,K5
1,2.097122,test,30,<bos> K3 V1 ; K2 V3 ; K1 V2 ; Q K1 ->,V2,K1
2,2.095568,test,29,<bos> K3 V1 ; K0 V2 ; K5 V0 ; Q K5 ->,V0,K5
3,1.987184,test,78,<bos> K7 V2 ; K0 V6 ; K5 V0 ; Q K5 ->,V0,K5
4,1.952963,test,128,<bos> K2 V6 ; K4 V3 ; K5 V0 ; Q K5 ->,V0,K5


### Feature `30` (support) top examples

,activation,split,index,prompt,target,query_key
0,2.361567,test,10,<bos> K3 V6 ; K2 V2 ; K0 V0 ; Q K3 ->,V6,K3
1,2.360566,test,114,<bos> K6 V6 ; K0 V2 ; K3 V1 ; Q K3 ->,V1,K3
2,2.357913,test,248,<bos> K3 V2 ; K0 V6 ; K1 V1 ; Q K3 ->,V2,K3
3,2.357209,test,168,<bos> K6 V4 ; K7 V5 ; K3 V2 ; Q K3 ->,V2,K3
4,2.356862,test,26,<bos> K3 V6 ; K7 V2 ; K1 V3 ; Q K3 ->,V6,K3


...

,correct_value_position,num_examples,feature_40_activation,feature_30_activation,feature_370_activation
0,2,150,0.017420,0.329751,0.411993
1,5,164,0.214219,0.365218,0.314802
2,8,186,0.399912,0.311489,0.307844


Grouped by query_pair_index


,query_pair_index,num_examples,feature_40_activation,feature_30_activation,feature_370_activation
0,0,150,0.017420,0.329751,0.411993
1,1,164,0.214219,0.365218,0.314802
2,2,186,0.399912,0.311489,0.307844


Grouped by corrupt_target


,corrupt_target,num_examples,feature_40_activation,feature_30_activation,feature_370_activation
0,V0,56,0.289281,0.268318,0.318563
1,V1,53,0.191673,0.282905,0.413627
2,V2,71,0.271656,0.345292,0.274597
3,V3,70,0.179846,0.386561,0.442786
4,V4,63,0.261538,0.233055,0.301969
5,V5,52,0.118830,0.358323,0.338313
6,V6,70,0.281960,0.374163,0.320684
7,V7,65,0.176922,0.402981,0.328739


## Feature Local Causes And Single-Feature Interventions

This cell answers two different questions.

1. local encoder cause:
   which parts of the `L1H0` vector push a feature on or off?
2. causal behavior:
   what happens if that feature is ablated on the clean prompt or patched from clean into the corrupt prompt?

Why this helps:

- the encoder contribution table is the feature-level analogue of neuron read decomposition
- the intervention table shows whether a feature actually changes `L2H0.Q`, attention, and margin


In [12]:
def build_modified_source(base_source_tensor: torch.Tensor, modified_vector: torch.Tensor, position_index: int) -> torch.Tensor:
    updated = base_source_tensor.clone()
    updated[0, position_index, :] = modified_vector.to(updated.dtype)
    return updated

source_patch = {
    "kind": "head_resid",
    "layer_index": 0,
    "head_index": 0,
    "source_positions": [final_position_index],
}

encoder_summary_rows = []
for feature_index in selected_features:
    clean_encoder_df = build_feature_encoder_contribution_table(feature_sae, clean_l1h0_vector, feature_index)
    corrupt_encoder_df = build_feature_encoder_contribution_table(feature_sae, corrupt_l1h0_vector, feature_index)
    contribution_delta_df = clean_encoder_df[["dimension", "contribution"]].merge(
        corrupt_encoder_df[["dimension", "contribution"]],
        on="dimension",
        suffixes=("_clean", "_corrupt"),
    )
    contribution_delta_df["contribution_delta"] = (
        contribution_delta_df["contribution_clean"] - contribution_delta_df["contribution_corrupt"]
    )
    contribution_delta_df["abs_contribution_delta"] = contribution_delta_df["contribution_delta"].abs()

    clean_activation = float(feature_sae.encode(clean_l1h0_vector.unsqueeze(0))[0, feature_index].item())
    corrupt_activation = float(feature_sae.encode(corrupt_l1h0_vector.unsqueeze(0))[0, feature_index].item())
    encoder_summary_rows.append(
        {
            "feature_index": feature_index,
            "role": selected_feature_roles[feature_index],
            "clean_activation": clean_activation,
            "corrupt_activation": corrupt_activation,
            "activation_delta": clean_activation - corrupt_activation,
        }
    )
    display(Markdown(f"### Feature `{feature_index}` local encoder causes"))
    display(
        contribution_delta_df.sort_values("abs_contribution_delta", ascending=False).head(10).reset_index(drop=True)
    )

encoder_feature_summary_df = pd.DataFrame(encoder_summary_rows)
display(encoder_feature_summary_df)

single_rows = []
for feature_index in selected_features:
    clean_ablation = intervene_on_sae_features(
        feature_sae,
        clean_l1h0_vector,
        feature_indices=[feature_index],
        mode="ablate",
    )
    clean_ablation_result = score_feature_intervention(
        model=analysis_model,
        bundle=bundle,
        prompt=clean_prompt,
        target_token=clean_target,
        base_cache=clean_cache,
        source_patch=source_patch,
        modified_source_tensor=build_modified_source(clean_l1h0_source, clean_ablation["reconstructed"], final_position_index),
        destination_layer_index=1,
        device=DEVICE,
    )

    corrupt_patch = intervene_on_sae_features(
        feature_sae,
        corrupt_l1h0_vector,
        feature_indices=[feature_index],
        mode="patch",
        source_vector=clean_l1h0_vector,
    )
    corrupt_patch_result = score_feature_intervention(
        model=analysis_model,
        bundle=bundle,
        prompt=corrupt_prompt,
        target_token=clean_target,
        base_cache=corrupt_cache,
        source_patch=source_patch,
        modified_source_tensor=build_modified_source(corrupt_l1h0_source, corrupt_patch["reconstructed"], final_position_index),
        destination_layer_index=1,
        device=DEVICE,
    )

    for label, result, base_q, base_pattern, payload in [
        ("clean ablate", clean_ablation_result, clean_l2h0_q, clean_l2h0_pattern, clean_ablation),
        ("corrupt patch", corrupt_patch_result, corrupt_l2h0_q, corrupt_l2h0_pattern, corrupt_patch),
    ]:
        intervened_q = result["details"]["destination_cache"]["attention"]["q"][0, 0, final_position_index, :].detach().cpu()
        intervened_pattern = result["details"]["destination_cache"]["attention"]["pattern"][0, 0, final_position_index, :].detach().cpu()
        single_rows.append(
            {
                "feature_index": feature_index,
                "role": selected_feature_roles[feature_index],
                "intervention": label,
                "predicted_token": result["predicted_token"],
                "foil_token": result["foil_token"],
                "margin_vs_clean_target": result["margin"],
                "feature_base_activation": float(payload["base_features"][feature_index].item()),
                "feature_modified_activation": float(payload["modified_features"][feature_index].item()),
                "l2h0_q_delta_norm": float((intervened_q - base_q).norm().item()),
                "attention_on_clean_value": float(intervened_pattern[clean_value_position].item()),
                "attention_on_corrupt_value": float(intervened_pattern[corrupt_value_position].item()),
                "attention_shift_to_clean_value": float((intervened_pattern[clean_value_position] - base_pattern[clean_value_position]).item()),
                "attention_shift_from_corrupt_value": float((intervened_pattern[corrupt_value_position] - base_pattern[corrupt_value_position]).item()),
            }
        )

single_feature_df = pd.DataFrame(single_rows)
display(single_feature_df)


### Feature `40` local encoder causes

,dimension,contribution_clean,contribution_corrupt,contribution_delta,abs_contribution_delta
0,2,0.009747,-0.334225,0.343972,0.343972
1,27,0.027098,-0.269343,0.296440,0.296440
2,4,0.151371,-0.141227,0.292598,0.292598
3,15,0.070488,-0.176059,0.246547,0.246547
4,17,-0.055142,-0.297583,0.242441,0.242441
5,31,-0.040778,0.192955,-0.233733,0.233733
6,43,0.013053,-0.217974,0.231026,0.231026
7,5,0.001970,-0.218503,0.220472,0.220472
8,46,0.009591,-0.200427,0.210018,0.210018
9,13,0.100221,-0.064945,0.165166,0.165166


### Feature `30` local encoder causes

,dimension,contribution_clean,contribution_corrupt,contribution_delta,abs_contribution_delta
0,2,0.007835,-0.268659,0.276494,0.276494
1,35,-0.125697,0.124056,-0.249753,0.249753
2,10,-0.079773,0.159747,-0.239520,0.239520
3,17,-0.048233,-0.260299,0.212066,0.212066
4,38,0.006301,-0.187036,0.193338,0.193338
5,30,0.000813,0.175291,-0.174478,0.174478
6,44,0.090894,-0.077098,0.167992,0.167992
7,27,0.013475,-0.133937,0.147412,0.147412
8,18,-0.013124,0.130787,-0.143912,0.143912
9,39,-0.028338,0.107960,-0.136298,0.136298


### Feature `370` local encoder causes

,dimension,contribution_clean,contribution_corrupt,contribution_delta,abs_contribution_delta
0,38,-0.011959,0.354962,-0.366920,0.366920
1,2,0.008174,-0.280270,0.288444,0.288444
2,46,-0.011650,0.243453,-0.255104,0.255104
3,27,-0.023151,0.230118,-0.253269,0.253269
4,23,-0.041465,0.211130,-0.252595,0.252595
5,5,-0.001988,0.220511,-0.222499,0.222499
6,11,0.014132,0.233639,-0.219507,0.219507
7,18,-0.019739,0.196705,-0.216444,0.216444
8,15,0.058891,-0.147092,0.205983,0.205983
9,13,0.071103,-0.046076,0.117180,0.117180


,feature_index,role,clean_activation,corrupt_activation,activation_delta
0,40,support,0.252239,0.000000,0.252239
1,30,support,0.000000,0.000000,0.000000
2,370,control,0.163699,1.425334,-1.261635


,feature_index,role,intervention,predicted_token,foil_token,margin_vs_clean_target,feature_base_activation,feature_modified_activation,l2h0_q_delta_norm,attention_on_clean_value,attention_on_corrupt_value,attention_shift_to_clean_value,attention_shift_from_corrupt_value
0,40,support,clean ablate,V0,V2,22.211855,0.252239,0.000000,0.347358,0.753751,0.000337,0.028149,-0.000024
1,40,support,corrupt patch,V2,V2,-17.257388,0.000000,0.252239,0.096911,0.000431,0.939058,0.000043,-0.000918
2,30,support,clean ablate,V0,V2,22.270034,0.000000,0.000000,0.339337,0.760746,0.000264,0.035144,-0.000097
3,30,support,corrupt patch,V2,V2,-17.362578,0.000000,0.000000,0.032470,0.000397,0.940082,0.000009,0.000107
4,370,control,clean ablate,V0,V2,22.316537,0.163699,0.000000,0.421496,0.766209,0.000237,0.040607,-0.000125
5,370,control,corrupt patch,V2,V2,-17.697389,1.425334,0.163699,0.614728,0.000547,0.937833,0.000159,-0.002142


## Feature Batch Check, Lens, And Cross-Layer Feature Sets

This is the compact feature synthesis cell.

Why this helps:

- the batch check asks whether single features matter beyond one prompt
- the lens tables show what decoder directions point toward in token space
- the upstream/downstream set comparison connects `L1H0` features to `L2H0.Q` features


In [13]:
batch_rows = []
for feature_index in selected_features:
    clean_correct_count = 0
    clean_ablate_correct_count = 0
    corrupt_restore_count = 0
    clean_margin_values = []
    clean_ablate_margin_values = []
    corrupt_patch_margin_values = []

    for row in bundle.raw_splits["test"][:BATCH_LIMIT]:
        row_clean_prompt = row["prompt"]
        row_clean_target = row["target"]
        row_corrupt_prompt, _ = make_query_swap_prompt(row)
        row_final_position_index = len(row_clean_prompt.split()) - 1
        row_source_patch = {
            "kind": "head_resid",
            "layer_index": 0,
            "head_index": 0,
            "source_positions": [row_final_position_index],
        }

        row_clean_result, row_clean_cache = run_prompt(
            analysis_model,
            bundle,
            row_clean_prompt,
            DEVICE,
            expected_target=row_clean_target,
            return_cache=True,
        )
        row_corrupt_result, row_corrupt_cache = run_prompt(
            analysis_model,
            bundle,
            row_corrupt_prompt,
            DEVICE,
            expected_target=row_clean_target,
            return_cache=True,
        )
        if row_clean_cache is None or row_corrupt_cache is None:
            raise ValueError("Expected caches during feature batch check")

        row_clean_source = head_residual_contribution(analysis_model, row_clean_cache, layer_index=0, head_index=0)
        row_corrupt_source = head_residual_contribution(analysis_model, row_corrupt_cache, layer_index=0, head_index=0)
        row_clean_vector = row_clean_source[0, row_final_position_index].detach().cpu()
        row_corrupt_vector = row_corrupt_source[0, row_final_position_index].detach().cpu()

        row_clean_ablation = intervene_on_sae_features(feature_sae, row_clean_vector, [feature_index], mode="ablate")
        row_clean_ablation_result = score_feature_intervention(
            model=analysis_model,
            bundle=bundle,
            prompt=row_clean_prompt,
            target_token=row_clean_target,
            base_cache=row_clean_cache,
            source_patch=row_source_patch,
            modified_source_tensor=build_modified_source(row_clean_source, row_clean_ablation["reconstructed"], row_final_position_index),
            destination_layer_index=1,
            device=DEVICE,
        )

        row_corrupt_patch = intervene_on_sae_features(feature_sae, row_corrupt_vector, [feature_index], mode="patch", source_vector=row_clean_vector)
        row_corrupt_patch_result = score_feature_intervention(
            model=analysis_model,
            bundle=bundle,
            prompt=row_corrupt_prompt,
            target_token=row_clean_target,
            base_cache=row_corrupt_cache,
            source_patch=row_source_patch,
            modified_source_tensor=build_modified_source(row_corrupt_source, row_corrupt_patch["reconstructed"], row_final_position_index),
            destination_layer_index=1,
            device=DEVICE,
        )

        clean_correct_count += int(row_clean_result["predicted_token"] == row_clean_target)
        clean_ablate_correct_count += int(row_clean_ablation_result["predicted_token"] == row_clean_target)
        corrupt_restore_count += int(row_corrupt_patch_result["predicted_token"] == row_clean_target)
        clean_margin_values.append(row_clean_result["margin"])
        clean_ablate_margin_values.append(row_clean_ablation_result["margin"])
        corrupt_patch_margin_values.append(row_corrupt_patch_result["margin"])

    batch_rows.append(
        {
            "feature_index": feature_index,
            "role": selected_feature_roles[feature_index],
            "num_examples": BATCH_LIMIT,
            "clean_accuracy": clean_correct_count / BATCH_LIMIT,
            "clean_ablate_accuracy": clean_ablate_correct_count / BATCH_LIMIT,
            "corrupt_patch_restore_rate": corrupt_restore_count / BATCH_LIMIT,
            "mean_clean_margin": float(sum(clean_margin_values) / BATCH_LIMIT),
            "mean_clean_ablate_margin": float(sum(clean_ablate_margin_values) / BATCH_LIMIT),
            "mean_corrupt_patch_margin": float(sum(corrupt_patch_margin_values) / BATCH_LIMIT),
        }
    )

batch_feature_df = pd.DataFrame(batch_rows)
display(batch_feature_df)

def top_token_rows(logits: torch.Tensor, id_to_token: dict[int, str], top_k: int = 5) -> list[dict[str, object]]:
    top_logits, top_indices = torch.topk(logits, k=min(top_k, logits.shape[0]))
    return [
        {"token": id_to_token[int(index.item())], "logit": float(value.item())}
        for value, index in zip(top_logits, top_indices)
    ]

lens_rows = []
for label, vector in [
    ("clean L1H0 write", clean_l1h0_vector),
    ("corrupt L1H0 write", corrupt_l1h0_vector),
    ("clean-minus-corrupt L1H0 delta", clean_l1h0_vector - corrupt_l1h0_vector),
]:
    logits = residual_vector_to_logits(analysis_model, vector)
    lens_rows.append(
        {
            "object": label,
            "clean_target_logit": float(logits[bundle.token_to_id[clean_target]].item()),
            "corrupt_target_logit": float(logits[bundle.token_to_id[corrupt_target]].item()),
            "clean_minus_corrupt_target": float(
                (logits[bundle.token_to_id[clean_target]] - logits[bundle.token_to_id[corrupt_target]]).item()
            ),
            "top_tokens": top_token_rows(logits, bundle.id_to_token),
        }
    )

for feature_index in selected_features:
    decoder_vector = feature_sae.decoder.weight[:, feature_index].detach().cpu()
    decoder_logits = residual_vector_to_logits(analysis_model, decoder_vector)
    clean_activation = float(feature_sae.encode(clean_l1h0_vector.unsqueeze(0))[0, feature_index].item())
    corrupt_activation = float(feature_sae.encode(corrupt_l1h0_vector.unsqueeze(0))[0, feature_index].item())
    clean_feature_vector = decoder_vector * clean_activation
    corrupt_feature_vector = decoder_vector * corrupt_activation
    clean_feature_logits = residual_vector_to_logits(analysis_model, clean_feature_vector)
    corrupt_feature_logits = residual_vector_to_logits(analysis_model, corrupt_feature_vector)

    lens_rows.extend(
        [
            {
                "object": f"feature {feature_index} decoder direction",
                "clean_target_logit": float(decoder_logits[bundle.token_to_id[clean_target]].item()),
                "corrupt_target_logit": float(decoder_logits[bundle.token_to_id[corrupt_target]].item()),
                "clean_minus_corrupt_target": float(
                    (decoder_logits[bundle.token_to_id[clean_target]] - decoder_logits[bundle.token_to_id[corrupt_target]]).item()
                ),
                "top_tokens": top_token_rows(decoder_logits, bundle.id_to_token),
            },
            {
                "object": f"feature {feature_index} clean activated contribution",
                "clean_target_logit": float(clean_feature_logits[bundle.token_to_id[clean_target]].item()),
                "corrupt_target_logit": float(clean_feature_logits[bundle.token_to_id[corrupt_target]].item()),
                "clean_minus_corrupt_target": float(
                    (clean_feature_logits[bundle.token_to_id[clean_target]] - clean_feature_logits[bundle.token_to_id[corrupt_target]]).item()
                ),
                "top_tokens": top_token_rows(clean_feature_logits, bundle.id_to_token),
            },
            {
                "object": f"feature {feature_index} corrupt activated contribution",
                "clean_target_logit": float(corrupt_feature_logits[bundle.token_to_id[clean_target]].item()),
                "corrupt_target_logit": float(corrupt_feature_logits[bundle.token_to_id[corrupt_target]].item()),
                "clean_minus_corrupt_target": float(
                    (corrupt_feature_logits[bundle.token_to_id[clean_target]] - corrupt_feature_logits[bundle.token_to_id[corrupt_target]]).item()
                ),
                "top_tokens": top_token_rows(corrupt_feature_logits, bundle.id_to_token),
            },
        ]
    )

lens_df = pd.DataFrame(lens_rows)
display(lens_df)

upstream_feature_set = support_features[:2]
set_rows = []
for label, site_kind, feature_set in [
    ("L1H0 upstream set", "upstream", upstream_feature_set),
    ("L2H0.Q downstream set", "downstream", downstream_feature_set),
]:
    if site_kind == "upstream":
        clean_intervention = intervene_on_sae_features(feature_sae, clean_l1h0_vector, feature_set, mode="ablate")
        clean_result_set = score_feature_intervention(
            model=analysis_model,
            bundle=bundle,
            prompt=clean_prompt,
            target_token=clean_target,
            base_cache=clean_cache,
            source_patch=source_patch,
            modified_source_tensor=build_modified_source(clean_l1h0_source, clean_intervention["reconstructed"], final_position_index),
            destination_layer_index=1,
            device=DEVICE,
        )
        corrupt_intervention = intervene_on_sae_features(feature_sae, corrupt_l1h0_vector, feature_set, mode="patch", source_vector=clean_l1h0_vector)
        corrupt_result_set = score_feature_intervention(
            model=analysis_model,
            bundle=bundle,
            prompt=corrupt_prompt,
            target_token=clean_target,
            base_cache=corrupt_cache,
            source_patch=source_patch,
            modified_source_tensor=build_modified_source(corrupt_l1h0_source, corrupt_intervention["reconstructed"], final_position_index),
            destination_layer_index=1,
            device=DEVICE,
        )
        clean_query = clean_result_set["details"]["destination_cache"]["attention"]["q"][0, 0, final_position_index, :].detach().cpu()
        corrupt_query = corrupt_result_set["details"]["destination_cache"]["attention"]["q"][0, 0, final_position_index, :].detach().cpu()
        clean_attn = clean_result_set["details"]["destination_cache"]["attention"]["pattern"][0, 0, final_position_index, :].detach().cpu()
        corrupt_attn = corrupt_result_set["details"]["destination_cache"]["attention"]["pattern"][0, 0, final_position_index, :].detach().cpu()
    else:
        clean_intervention = intervene_on_sae_features(l2q_sae, clean_l2h0_q, feature_set, mode="ablate")
        clean_result_set = score_query_feature_intervention(
            model=analysis_model,
            bundle=bundle,
            prompt=clean_prompt,
            target_token=clean_target,
            base_cache=clean_cache,
            layer_index=1,
            head_index=0,
            position_index=final_position_index,
            modified_query_vector=clean_intervention["reconstructed"],
            device=DEVICE,
        )
        corrupt_intervention = intervene_on_sae_features(l2q_sae, corrupt_l2h0_q, feature_set, mode="patch", source_vector=clean_l2h0_q)
        corrupt_result_set = score_query_feature_intervention(
            model=analysis_model,
            bundle=bundle,
            prompt=corrupt_prompt,
            target_token=clean_target,
            base_cache=corrupt_cache,
            layer_index=1,
            head_index=0,
            position_index=final_position_index,
            modified_query_vector=corrupt_intervention["reconstructed"],
            device=DEVICE,
        )
        clean_query = clean_result_set["details"]["modified_q"][0, 0, final_position_index, :].detach().cpu()
        corrupt_query = corrupt_result_set["details"]["modified_q"][0, 0, final_position_index, :].detach().cpu()
        clean_attn = clean_result_set["details"]["pattern"][0, 0, final_position_index, :].detach().cpu()
        corrupt_attn = corrupt_result_set["details"]["pattern"][0, 0, final_position_index, :].detach().cpu()

    set_rows.append(
        {
            "site_label": label,
            "feature_set": feature_set,
            "clean_ablate_predicted_token": clean_result_set["predicted_token"],
            "clean_ablate_margin": clean_result_set["margin"],
            "clean_ablate_q_delta_norm": float((clean_query - clean_l2h0_q).norm().item()),
            "clean_ablate_attention_on_clean_value": float(clean_attn[clean_value_position].item()),
            "corrupt_patch_predicted_token": corrupt_result_set["predicted_token"],
            "corrupt_patch_margin_vs_clean_target": corrupt_result_set["margin"],
            "corrupt_patch_q_delta_norm": float((corrupt_query - corrupt_l2h0_q).norm().item()),
            "corrupt_patch_attention_on_clean_value": float(corrupt_attn[clean_value_position].item()),
            "corrupt_patch_attention_on_corrupt_value": float(corrupt_attn[corrupt_value_position].item()),
        }
    )

feature_set_df = pd.DataFrame(set_rows)
display(feature_set_df)


,feature_index,role,num_examples,clean_accuracy,clean_ablate_accuracy,corrupt_patch_restore_rate,mean_clean_margin,mean_clean_ablate_margin,mean_corrupt_patch_margin
0,40,support,64,0.953125,0.953125,0.015625,11.664123,11.682202,-17.375041
1,30,support,64,0.953125,0.953125,0.015625,11.664123,11.726490,-17.388819
2,370,control,64,0.953125,0.953125,0.015625,11.664123,11.777029,-17.360223


,object,clean_target_logit,corrupt_target_logit,clean_minus_corrupt_target,top_tokens
0,clean L1H0 write,-5.623686,2.243749,-7.867435,"[{'token': 'V7', 'logit': 10.048322677612305}, {'token': 'K6', 'logit': 6.604335784912109}, {'token': 'K4', 'logit': 4.113252639770508}, {'token': 'V1', 'logit': 3.8423404693603516}, {'token': 'V2..."
1,corrupt L1H0 write,-4.167481,-1.146262,-3.021219,"[{'token': 'K7', 'logit': 9.117042541503906}, {'token': '<bos>', 'logit': 4.734622001647949}, {'token': 'K1', 'logit': 4.721177101135254}, {'token': 'K3', 'logit': 4.626852035522461}, {'token': 'K..."
2,clean-minus-corrupt L1H0 delta,2.026600,1.763520,0.263080,"[{'token': 'K0', 'logit': 6.839349746704102}, {'token': 'V1', 'logit': 4.927506446838379}, {'token': 'V7', 'logit': 4.726060390472412}, {'token': 'V6', 'logit': 4.323816776275635}, {'token': 'V4',..."
3,feature 40 decoder direction,4.121844,1.034610,3.087234,"[{'token': 'V5', 'logit': 7.171279430389404}, {'token': 'Q', 'logit': 5.237597942352295}, {'token': 'V0', 'logit': 4.1218438148498535}, {'token': 'V1', 'logit': 3.7430121898651123}, {'token': 'K0'..."
4,feature 40 clean activated contribution,4.120389,1.034244,3.086144,"[{'token': 'V5', 'logit': 7.168747901916504}, {'token': 'Q', 'logit': 5.235748767852783}, {'token': 'V0', 'logit': 4.120388507843018}, {'token': 'V1', 'logit': 3.7416911125183105}, {'token': 'K0',..."
5,feature 40 corrupt activated contribution,0.000000,0.000000,0.000000,"[{'token': '<bos>', 'logit': 0.0}, {'token': ';', 'logit': 0.0}, {'token': 'Q', 'logit': 0.0}, {'token': '->', 'logit': 0.0}, {'token': 'K0', 'logit': 0.0}]"
6,feature 30 decoder direction,4.292377,-2.278683,6.571059,"[{'token': 'K1', 'logit': 9.63432502746582}, {'token': 'V3', 'logit': 7.724791526794434}, {'token': 'K4', 'logit': 6.822149276733398}, {'token': 'V5', 'logit': 4.542654037475586}, {'token': 'K7', ..."
7,feature 30 clean activated contribution,0.000000,0.000000,0.000000,"[{'token': '<bos>', 'logit': 0.0}, {'token': ';', 'logit': 0.0}, {'token': 'Q', 'logit': 0.0}, {'token': '->', 'logit': 0.0}, {'token': 'K0', 'logit': 0.0}]"
8,feature 30 corrupt activated contribution,0.000000,0.000000,0.000000,"[{'token': '<bos>', 'logit': 0.0}, {'token': ';', 'logit': 0.0}, {'token': 'Q', 'logit': 0.0}, {'token': '->', 'logit': 0.0}, {'token': 'K0', 'logit': 0.0}]"
9,feature 370 decoder direction,-3.772310,-0.599122,-3.173188,"[{'token': '<bos>', 'logit': 7.488486289978027}, {'token': 'V6', 'logit': 6.376976013183594}, {'token': 'K2', 'logit': 3.0376853942871094}, {'token': 'K3', 'logit': 1.4523577690124512}, {'token': ..."


,site_label,feature_set,clean_ablate_predicted_token,clean_ablate_margin,clean_ablate_q_delta_norm,clean_ablate_attention_on_clean_value,corrupt_patch_predicted_token,corrupt_patch_margin_vs_clean_target,corrupt_patch_q_delta_norm,corrupt_patch_attention_on_clean_value,corrupt_patch_attention_on_corrupt_value
0,L1H0 upstream set,"[40, 30]",V0,22.211855,0.347358,0.753751,V2,-17.257388,0.096911,0.000431,0.939058
1,L2H0.Q downstream set,"[93, 37]",V0,22.102778,0.469322,0.738782,V2,-17.457297,0.341313,0.000299,0.944780


## Neuron Dashboard And Upstream Causes

This cell starts from the strongest `L1` MLP neurons on the anchor prompt and traces what drives them.

Why this helps:

- it identifies neurons that matter by exact single-neuron ablation, not by raw activation size
- it compares clean vs corrupt activation
- it asks which upstream heads and source tokens inside those heads move the neuron back toward clean


In [14]:
single_prompt_l1_df = build_mlp_neuron_contribution_table(
    model=analysis_model,
    bundle=bundle,
    prompt=clean_prompt,
    cache=clean_cache,
    layer_index=0,
    target_token=clean_target,
    foil_token=foil_token,
    device=DEVICE,
    position=final_position_index,
    top_k=5,
    include_exact_ablation=True,
)
selected_neurons = (
    single_prompt_l1_df
    .sort_values(["exact_ablation_margin_drop", "activated"], ascending=[False, False])
    .head(5)["neuron_index"]
    .astype(int)
    .tolist()
)
display(
    single_prompt_l1_df
    .sort_values(["exact_ablation_margin_drop", "activated"], ascending=[False, False])
    .head(5)
    .reset_index(drop=True)
)

clean_activated = clean_cache["blocks"][0]["mlp"]["activated"][0, -1, :].detach().cpu()
corrupt_activated = corrupt_cache["blocks"][0]["mlp"]["activated"][0, -1, :].detach().cpu()
shift_rows = []
for neuron_index in selected_neurons:
    clean_value = float(clean_activated[neuron_index].item())
    corrupt_value = float(corrupt_activated[neuron_index].item())
    shift_rows.append(
        {
            "neuron_index": neuron_index,
            "clean_activation": clean_value,
            "corrupt_activation": corrupt_value,
            "clean_minus_corrupt": clean_value - corrupt_value,
        }
    )
shift_df = pd.DataFrame(shift_rows).sort_values("clean_minus_corrupt", ascending=False).reset_index(drop=True)
display(shift_df)

head_effect_df = build_mlp_neuron_upstream_head_effect_table(
    model=analysis_model,
    prompt=clean_prompt,
    cache=clean_cache,
    layer_index=0,
    neuron_indices=selected_neurons,
    position=-1,
)
head_effect_summary_df = (
    head_effect_df
    .groupby(["head_index", "component"], as_index=False)
    .agg(
        mean_abs_activation_delta=("abs_activation_delta", "mean"),
        max_abs_activation_delta=("abs_activation_delta", "max"),
        mean_signed_activation_delta=("activation_delta", "mean"),
    )
    .sort_values(["mean_abs_activation_delta", "max_abs_activation_delta"], ascending=[False, False])
    .reset_index(drop=True)
)
display(head_effect_summary_df)

head_patch_df = build_mlp_neuron_clean_corrupt_head_patch_table(
    model=analysis_model,
    clean_prompt=clean_prompt,
    clean_cache=clean_cache,
    corrupt_prompt=corrupt_prompt,
    corrupt_cache=corrupt_cache,
    layer_index=0,
    neuron_indices=selected_neurons,
    position=-1,
)
head_patch_summary_df = (
    head_patch_df
    .groupby(["head_index", "component"], as_index=False)
    .agg(
        mean_recovery_toward_clean=("recovery_toward_clean", "mean"),
        max_recovery_toward_clean=("recovery_toward_clean", "max"),
    )
    .sort_values(["mean_recovery_toward_clean", "max_recovery_toward_clean"], ascending=[False, False])
    .reset_index(drop=True)
)
dominant_upstream_head = int(head_patch_summary_df.iloc[0]["head_index"])
display(head_patch_summary_df)

source_patch_df = build_mlp_neuron_clean_corrupt_source_patch_table(
    model=analysis_model,
    clean_prompt=clean_prompt,
    clean_cache=clean_cache,
    corrupt_prompt=corrupt_prompt,
    corrupt_cache=corrupt_cache,
    layer_index=0,
    head_index=dominant_upstream_head,
    neuron_indices=selected_neurons,
    position=-1,
)
source_patch_summary_df = (
    source_patch_df
    .groupby(["source_position", "clean_source_token", "corrupt_source_token"], as_index=False)
    .agg(
        mean_recovery_toward_clean=("recovery_toward_clean", "mean"),
        max_recovery_toward_clean=("recovery_toward_clean", "max"),
    )
    .sort_values(["mean_recovery_toward_clean", "max_recovery_toward_clean"], ascending=[False, False])
    .reset_index(drop=True)
)
display(source_patch_summary_df)


,neuron_index,gate,up,activated,write_norm,standalone_target_logit,standalone_foil_logit,standalone_target_minus_foil,top_token,top_k_tokens,exact_ablation_predicted_token,exact_ablation_margin,exact_ablation_margin_drop,exact_ablation_correct
0,2,1.447725,1.312805,1.538801,1.492506,-1.890824,-12.137550,10.246726,K5,"[{'token': 'K5', 'logit': 9.435081481933594}, {'token': 'K3', 'logit': 7.731704235076904}, {'token': 'K7', 'logit': 3.54988431930542}, {'token': '<bos>', 'logit': 2.709653854370117}, {'token': 'K0...",V0,20.586555,1.450418,True
1,60,1.723909,-1.648986,-2.412408,2.334694,3.613064,-4.266998,7.880062,V3,"[{'token': 'V3', 'logit': 13.141498565673828}, {'token': 'V7', 'logit': 8.004819869995117}, {'token': 'V5', 'logit': 6.269511699676514}, {'token': 'V1', 'logit': 4.405825614929199}, {'token': 'V4'...",V0,21.013517,1.023456,True
2,30,0.557064,2.331540,0.825753,0.820388,-0.689578,-5.825058,5.135480,V7,"[{'token': 'V7', 'logit': 9.942569732666016}, {'token': 'V1', 'logit': 8.211318969726562}, {'token': 'K4', 'logit': 6.869964599609375}, {'token': 'K1', 'logit': 6.226964473724365}, {'token': 'V5',...",V0,21.481939,0.555034,True
3,29,-0.874758,1.320955,-0.340028,0.347986,-4.886423,-9.502146,4.615723,V5,"[{'token': 'V5', 'logit': 15.767328262329102}, {'token': ';', 'logit': 8.283942222595215}, {'token': '->', 'logit': 7.393098831176758}, {'token': '<bos>', 'logit': 4.241184234619141}, {'token': 'K...",V0,21.792087,0.244886,True
4,20,-0.362683,2.769741,-0.412173,0.414871,5.055267,-1.205655,6.260922,V3,"[{'token': 'V3', 'logit': 13.720274925231934}, {'token': '<bos>', 'logit': 9.732295036315918}, {'token': 'K1', 'logit': 9.15096664428711}, {'token': 'K4', 'logit': 9.054460525512695}, {'token': 'V...",V0,21.877111,0.159862,True


,neuron_index,clean_activation,corrupt_activation,clean_minus_corrupt
0,2,1.538801,0.190959,1.347842
1,30,0.825753,-0.293172,1.118924
2,29,-0.340028,0.005245,-0.345273
3,20,-0.412173,0.598298,-1.010470
4,60,-2.412408,0.293758,-2.706165


,head_index,component,mean_abs_activation_delta,max_abs_activation_delta,mean_signed_activation_delta
0,1,L1H1,0.54759,1.538843,-0.184614
1,0,L1H0,0.51578,0.955640,-0.189871


,head_index,component,mean_recovery_toward_clean,max_recovery_toward_clean
0,0,L1H0,1.269161,2.629084
1,1,L1H1,-0.002565,0.011530


,source_position,clean_source_token,corrupt_source_token,mean_recovery_toward_clean,max_recovery_toward_clean
0,11,K4,K7,0.847183,2.236633
1,9,;,;,0.021611,0.061791
2,6,;,;,0.018556,0.053119
3,3,;,;,0.015402,0.044146
4,5,V0,V0,0.003808,0.038337
5,12,->,->,0.001151,0.006967
6,0,<bos>,<bos>,0.000512,0.001624
7,2,V2,V2,-0.000145,0.000403
8,1,K7,K7,-0.000351,0.000537
9,10,Q,Q,-0.000933,0.000017


## Neuron Read Decomposition And Batch Neuron Checks

This is the compact neuron synthesis cell.

Why this helps:

- the read decomposition shows which input dimensions move `gate` and `up`
- the grouped label tables test whether the selected neurons are query-like, value-like, or positional
- the batch ablation table checks whether these neurons stay important beyond one prompt


In [15]:
for neuron_index in selected_neurons[:3]:
    print(f"Neuron {neuron_index}")
    display(
        build_mlp_neuron_read_comparison_table(
            model=analysis_model,
            clean_cache=clean_cache,
            corrupt_cache=corrupt_cache,
            layer_index=0,
            neuron_index=neuron_index,
            position=-1,
        ).head(12)
    )

neuron_activation_df = collect_mlp_neuron_activation_table(
    model=analysis_model,
    bundle=bundle,
    split="test",
    layer_index=0,
    device=DEVICE,
    neuron_indices=selected_neurons,
    position=-1,
)

for group_column in ["query_key", "target", "correct_value_position", "query_pair_index", "corrupt_target"]:
    print(f"Grouped by {group_column}")
    display(build_mlp_neuron_group_summary_table(neuron_activation_df, selected_neurons, group_column))

top_examples = build_top_mlp_neuron_examples(
    neuron_activation_df,
    neuron_indices=selected_neurons,
    top_k=5,
)
for neuron_index in selected_neurons:
    display(Markdown(f"### Neuron `{neuron_index}` top prompts"))
    display(pd.DataFrame(top_examples[neuron_index]))

batch_ablation_df = build_mlp_neuron_batch_ablation_table(
    model=analysis_model,
    bundle=bundle,
    split="test",
    layer_index=0,
    device=DEVICE,
    neuron_indices=selected_neurons,
    limit=NEURON_BATCH_LIMIT,
    position=-1,
)
display(batch_ablation_df.sort_values("mean_margin_drop", ascending=False).reset_index(drop=True))


Neuron 2


,input_dim,clean_x_value,corrupt_x_value,x_delta,gate_weight,clean_gate_contribution,corrupt_gate_contribution,gate_contribution_delta,up_weight,clean_up_contribution,corrupt_up_contribution,up_contribution_delta,combined_abs_delta
0,46,-1.604312,-0.161402,-1.442910,-0.261132,0.418937,0.042147,0.376790,-0.448715,0.719880,0.072424,0.647456,1.024246
1,38,-0.342968,-1.157577,0.814610,0.195041,-0.066893,-0.225775,0.158882,-0.352615,0.120936,0.408179,-0.287244,0.446126
2,35,1.095117,-0.112780,1.207897,-0.142490,-0.156043,0.016070,-0.172113,-0.190798,-0.208947,0.021518,-0.230465,0.402578
3,27,0.739467,-1.085115,1.824582,-0.098162,-0.072588,0.106517,-0.179105,0.096992,0.071723,-0.105248,0.176970,0.356076
4,0,-0.337787,0.398055,-0.735843,0.227944,-0.076997,0.090734,-0.167731,-0.197136,0.066590,-0.078471,0.145061,0.312792
5,43,-0.592688,0.701212,-1.293900,0.200941,-0.119096,0.140903,-0.259998,-0.029661,0.017580,-0.020799,0.038379,0.298377
6,24,-1.324461,-0.026213,-1.298249,-0.119626,0.158439,0.003136,0.155304,-0.105656,0.139937,0.002770,0.137167,0.292471
7,10,0.044235,-1.262388,1.306623,-0.106764,-0.004723,0.134777,-0.139500,0.112131,0.004960,-0.141552,0.146512,0.286012
8,15,-0.308561,0.620483,-0.929044,0.239302,-0.073839,0.148483,-0.222322,0.030575,-0.009434,0.018971,-0.028405,0.250727
9,7,-0.881069,-0.035363,-0.845706,-0.074778,0.065885,0.002644,0.063241,0.198084,-0.174526,-0.007005,-0.167521,0.230762


Neuron 60


,input_dim,clean_x_value,corrupt_x_value,x_delta,gate_weight,clean_gate_contribution,corrupt_gate_contribution,gate_contribution_delta,up_weight,clean_up_contribution,corrupt_up_contribution,up_contribution_delta,combined_abs_delta
0,43,-0.592688,0.701212,-1.293900,-0.230488,0.136607,-0.161621,0.298228,0.245582,-0.145554,0.172205,-0.317759,0.615988
1,27,0.739467,-1.085115,1.824582,0.249798,0.184717,-0.271060,0.455777,-0.031697,-0.023439,0.034395,-0.057833,0.513611
2,7,-0.881069,-0.035363,-0.845706,-0.328587,0.289508,0.011620,0.277888,0.243985,-0.214968,-0.008628,-0.206340,0.484228
3,5,-0.534221,0.536962,-1.071183,-0.238477,0.127399,-0.128053,0.255452,0.165039,-0.088167,0.088620,-0.176787,0.432239
4,46,-1.604312,-0.161402,-1.442910,-0.192894,0.309462,0.031134,0.278329,-0.105144,0.168684,0.016971,0.151714,0.430042
5,4,0.042022,0.846911,-0.804889,-0.204397,-0.008589,-0.173106,0.164517,-0.316048,-0.013281,-0.267664,0.254383,0.418900
6,10,0.044235,-1.262388,1.306623,-0.061912,-0.002739,0.078157,-0.080896,-0.252474,-0.011168,0.318720,-0.329888,0.410784
7,14,-0.835326,-0.274547,-0.560780,-0.258740,0.216132,0.071036,0.145096,0.434052,-0.362575,-0.119168,-0.243408,0.388504
8,31,0.635200,-0.268761,0.903962,0.362699,0.230386,-0.097479,0.327866,-0.065066,-0.041330,0.017487,-0.058817,0.386683
9,23,-0.056166,-0.903655,0.847489,-0.077651,0.004361,0.070170,-0.065808,0.299660,-0.016831,-0.270789,0.253959,0.319767


Neuron 30


...

### Neuron `29` top prompts

,split,index,prompt,target,query_key,activation
0,test,451,<bos> K3 V2 ; K5 V0 ; K1 V3 ; Q K1 ->,V3,K1,4.466382
1,test,374,<bos> K5 V6 ; K2 V0 ; K0 V3 ; Q K0 ->,V3,K0,3.694309
2,test,225,<bos> K3 V6 ; K1 V0 ; K7 V3 ; Q K1 ->,V0,K1,3.002172
3,test,408,<bos> K3 V0 ; K2 V6 ; K4 V3 ; Q K4 ->,V3,K4,2.846335
4,test,483,<bos> K3 V0 ; K5 V3 ; K1 V1 ; Q K1 ->,V1,K1,2.303285


### Neuron `20` top prompts

,split,index,prompt,target,query_key,activation
0,test,4,<bos> K7 V0 ; K6 V2 ; K2 V6 ; Q K6 ->,V2,K6,4.371105
1,test,431,<bos> K7 V0 ; K6 V6 ; K1 V3 ; Q K6 ->,V6,K6,4.154695
2,test,178,<bos> K6 V4 ; K5 V3 ; K1 V6 ; Q K6 ->,V4,K6,3.998068
3,test,2,<bos> K6 V1 ; K3 V0 ; K5 V6 ; Q K6 ->,V1,K6,3.606292
4,test,445,<bos> K1 V0 ; K6 V3 ; K2 V2 ; Q K6 ->,V3,K6,3.503217


,neuron_index,num_examples,clean_accuracy,ablate_accuracy,mean_margin,mean_ablate_margin,mean_margin_drop,prediction_change_rate
0,29,64,0.953125,0.953125,11.664123,11.629597,0.034526,0.0
1,2,64,0.953125,0.953125,11.664123,11.641702,0.022421,0.0
2,30,64,0.953125,0.953125,11.664123,11.676194,-0.012071,0.0
3,60,64,0.953125,0.953125,11.664123,11.706204,-0.042081,0.0
4,20,64,0.953125,0.953125,11.664123,11.817362,-0.153239,0.0


## Register And Stage Summary

This final section is the synthesis layer.

It does **not** magically solve the internal algorithm. It simply compresses the previous tracking objects into the best current register candidates:

- where query-key identity is most readable
- which heads behave most like selected-slot trackers
- where selected-value identity becomes strongly readable
- which head writes the strongest selected-value evidence


In [16]:
stage_margin_df = build_layer_feature_readout_table(
    model=analysis_model,
    bundle=bundle,
    cache=clean_cache,
    target_token=clean_target,
    foil_token=foil_token,
)
stage_variable_df = build_stage_variable_readout_table(
    model=analysis_model,
    bundle=bundle,
    cache=clean_cache,
    row=example,
)
display(stage_margin_df[["stage", "top_token", "target_minus_foil", "top_k_tokens"]])
display(stage_variable_df)

single_prompt_head_df = build_single_prompt_head_role_table(
    model=analysis_model,
    bundle=bundle,
    row=example,
    cache=clean_cache,
    target_token=clean_target,
    foil_token=foil_token,
)
single_prompt_slot_df = build_single_prompt_slot_routing_table(
    model=analysis_model,
    row=example,
    cache=clean_cache,
)
query_swap_head_df = build_query_swap_head_role_comparison_table(
    model=analysis_model,
    clean_row=example,
    clean_cache=clean_cache,
    corrupt_row=corrupt_row,
    corrupt_cache=corrupt_cache,
)
query_swap_slot_df = build_query_swap_slot_routing_comparison_table(
    model=analysis_model,
    clean_row=example,
    clean_cache=clean_cache,
    corrupt_row=corrupt_row,
    corrupt_cache=corrupt_cache,
)
query_swap_slot_df["clean_and_corrupt_top_slot_match_selected"] = (
    query_swap_slot_df["clean_top_slot_matches_selected"] & query_swap_slot_df["corrupt_top_slot_matches_selected"]
)

batch_stage_variable_df = collect_stage_variable_readout_table(
    model=analysis_model,
    bundle=bundle,
    split="test",
    device=DEVICE,
    limit=BATCH_LIMIT,
)
stage_summary_df = build_stage_variable_summary_table(batch_stage_variable_df)
stage_order = stage_variable_df["stage"].tolist()
stage_summary_df["stage"] = pd.Categorical(stage_summary_df["stage"], categories=stage_order, ordered=True)
stage_summary_df = stage_summary_df.sort_values("stage").reset_index(drop=True)

batch_head_role_df = collect_head_role_attention_table(
    model=analysis_model,
    bundle=bundle,
    split="test",
    device=DEVICE,
    limit=BATCH_LIMIT,
)
head_role_summary_df = build_head_role_summary_table(batch_head_role_df)

batch_slot_routing_df = collect_slot_routing_table(
    model=analysis_model,
    bundle=bundle,
    split="test",
    device=DEVICE,
    limit=BATCH_LIMIT,
)
slot_routing_summary_df = build_slot_routing_summary_table(batch_slot_routing_df)

display(stage_summary_df)
display(slot_routing_summary_df.sort_values(["top_selected_slot_rate", "mean_selected_slot_attention"], ascending=[False, False]).reset_index(drop=True))
display(head_role_summary_df.sort_values("mean_query_key_attention", ascending=False).reset_index(drop=True))

best_query_stage = stage_summary_df.sort_values(
    ["query_top_key_rate", "mean_query_minus_best_distractor_key"],
    ascending=[False, False],
).iloc[0]
best_selected_value_stage = stage_summary_df.sort_values(
    ["target_top_value_rate", "mean_target_minus_best_distractor_value"],
    ascending=[False, False],
).iloc[0]
best_query_head = head_role_summary_df.sort_values(
    ["mean_query_key_attention", "top_query_key_rate"],
    ascending=[False, False],
).iloc[0]
best_slot_head = slot_routing_summary_df.sort_values(
    ["top_selected_slot_rate", "mean_selected_slot_attention"],
    ascending=[False, False],
).iloc[0]
best_value_head = single_prompt_head_df.sort_values(
    ["selected_value_weighted_target_minus_foil", "selected_value_attention"],
    ascending=[False, False],
).iloc[0]

register_map_df = pd.DataFrame(
    [
        {
            "register": "query_key_reader",
            "best_candidate_site": best_query_head["component"],
            "evidence": f"mean_query_key_attention={best_query_head['mean_query_key_attention']:.3f}, top_query_key_rate={best_query_head['top_query_key_rate']:.3f}",
        },
        {
            "register": "selected_slot_tracker",
            "best_candidate_site": best_slot_head["component"],
            "evidence": f"top_selected_slot_rate={best_slot_head['top_selected_slot_rate']:.3f}, mean_selected_slot_attention={best_slot_head['mean_selected_slot_attention']:.3f}",
        },
        {
            "register": "selected_value_readout",
            "best_candidate_site": best_selected_value_stage["stage"],
            "evidence": f"target_top_value_rate={best_selected_value_stage['target_top_value_rate']:.3f}, mean_target_minus_best_distractor_value={best_selected_value_stage['mean_target_minus_best_distractor_value']:.3f}",
        },
        {
            "register": "selected_value_writer",
            "best_candidate_site": best_value_head["component"],
            "evidence": f"selected_value_weighted_target_minus_foil={best_value_head['selected_value_weighted_target_minus_foil']:.3f}",
        },
    ]
)
display(register_map_df)


,stage,top_token,target_minus_foil,top_k_tokens
0,L1 resid_in,->,-1.796998,"[->, V4, <bos>, V5, K0]"
1,L1 after attn,->,1.616020,"[->, V7, <bos>, V4, V0]"
2,L1 after mlp,V7,1.494648,"[V7, V4, ->, V0, V2]"
3,L2 resid_in,V7,1.494648,"[V7, V4, ->, V0, V2]"
4,L2 after attn,V0,22.462303,"[V0, V2, V4, Q, K4]"
5,L2 after mlp,V0,22.036968,"[V0, V2, V4, Q, K4]"


,stage,top_token,top_logit,top_key_token,top_key_logit,query_key_logit,query_is_top_key,query_minus_best_distractor_key,top_value_token,top_value_logit,target_value_logit,target_is_top_value,target_minus_best_distractor_value
0,L1 resid_in,->,28.118256,K0,3.253319,-1.266711,False,-0.083208,V4,9.337807,-0.316064,False,-1.796998
1,L1 after attn,->,13.015645,K6,1.402527,-1.879755,False,-3.282282,V7,10.684624,5.074450,False,-5.610174
2,L1 after mlp,V7,15.078071,K3,-0.034286,-0.439025,False,0.128615,V7,15.078071,6.725322,False,-8.352749
3,L2 resid_in,V7,15.078071,K3,-0.034286,-0.439025,False,0.128615,V7,15.078071,6.725322,False,-8.352749
4,L2 after attn,V0,29.535278,K4,4.952985,4.952985,True,1.271083,V0,29.535278,29.535278,True,22.462302
5,L2 after mlp,V0,29.739082,K4,5.260845,5.260845,True,2.483545,V0,29.739082,29.739082,True,22.036968


,stage,num_examples,query_top_key_rate,target_top_value_rate,mean_query_minus_best_distractor_key,mean_target_minus_best_distractor_value
0,L1 resid_in,64,0.125000,0.109375,-1.240012,-3.181544
1,L1 after attn,64,0.171875,0.171875,-1.425321,-4.377169
2,L1 after mlp,64,0.156250,0.093750,-1.595604,-4.083196
3,L2 resid_in,64,0.156250,0.093750,-1.595604,-4.083196
4,L2 after attn,64,0.093750,0.953125,-2.064245,11.369819
5,L2 after mlp,64,0.078125,0.953125,-1.966049,13.289566


,layer_index,head_index,component,num_examples,top_selected_slot_rate,mean_selected_slot_attention,mean_selected_slot_key_attention,mean_selected_slot_value_attention,mean_top_slot_attention
0,1,0,L2H0,64,0.781250,0.664354,0.004907,0.293667,0.803660
1,1,1,L2H1,64,0.734375,0.495320,0.064614,0.287100,0.554926
2,0,1,L1H1,64,0.312500,0.259954,0.092830,0.153439,0.393205
3,0,0,L1H0,64,0.296875,0.224733,0.163041,0.020910,0.377393


,layer_index,head_index,component,num_examples,mean_query_key_attention,mean_selected_key_attention,mean_selected_value_attention,mean_distractor_key_attention,mean_distractor_value_attention,mean_separator_attention,mean_query_marker_attention,mean_answer_marker_attention,top_query_key_rate,top_selected_key_rate,top_selected_value_rate,top_distractor_key_rate,top_distractor_value_rate
0,0,0,L1H0,64,0.459685,0.163041,0.020910,0.145725,0.075131,0.113624,0.005535,0.013796,0.46875,0.234375,0.000000,0.171875,0.078125
1,1,1,L2H1,64,0.344233,0.064614,0.287100,0.100627,0.009375,0.148497,0.015508,0.029902,0.37500,0.062500,0.296875,0.109375,0.000000
2,0,1,L1H1,64,0.145883,0.092830,0.153439,0.188454,0.317677,0.040503,0.030887,0.022687,0.18750,0.062500,0.171875,0.187500,0.390625
3,1,0,L2H0,64,0.051185,0.004907,0.293667,0.062970,0.105398,0.410925,0.055426,0.015006,0.06250,0.000000,0.328125,0.031250,0.109375


,register,best_candidate_site,evidence
0,query_key_reader,L1H0,"mean_query_key_attention=0.460, top_query_key_rate=0.469"
1,selected_slot_tracker,L2H0,"top_selected_slot_rate=0.781, mean_selected_slot_attention=0.664"
2,selected_value_readout,L2 after mlp,"target_top_value_rate=0.953, mean_target_minus_best_distractor_value=13.290"
3,selected_value_writer,L2H0,selected_value_weighted_target_minus_foil=24.516


## What This Notebook Now Gives You

This unified notebook does not claim the internal algorithm is solved. What it *does* give you is one canonical analysis artifact with no split across four notebook styles.

It now contains:

- activation patching
- path patching
- causal ablations
- QK / OV analysis
- circuit tracing
- sparse autoencoder feature dashboards
- feature logit lens and cross-layer feature-set checks
- neuron-level upstream-cause and read-decomposition tables
- a register-style summary at the end

The remaining gap is no longer "where is the data?" but "how do we turn these tracked objects into a minimal symbolic program?"
